# NEAT1 ENE+A-rich triplex: computational pipeline

Google Colab notebook for sequence retrieval, folding, homolog analysis, 3D structure prediction, MD, pocket detection and docking. Cells are in the order they were run (including a restart cell for the extended MD run after Colab disconnects). trRosettaRNA and RNAComposer were run on their web servers. Set `Entrez.email` to your own address before using NCBI services.

In [ ]:
!pip install biopython

## 1. Target sequence retrieval (NCBI)

In [ ]:
from Bio import Entrez, SeqIO

Entrez.email = "your_email@example.com"

accession_ids = {
    "NEAT1_v1": "NR_131012",
    "NEAT1_v2": "NR_003491"
}

sequences = {}

for name, acc in accession_ids.items():
    handle = Entrez.efetch(db="nucleotide", id=acc, rettype="fasta", retmode="text")
    record = SeqIO.read(handle, "fasta")
    handle.close()
    sequences[name] = record

In [ ]:
for name, record in sequences.items():
    filename = f"{name}.fasta"
    with open(filename, "w") as f:
        f.write(f">{name}|{record.id}\n")
        f.write(str(record.seq) + "\n")

import os
for f in os.listdir("."):
    if f.endswith(".fasta"):
        print(" -", f)

In [ ]:
import os

old_files = ["NEAT1_v2.fasta"]
for f in old_files:
    if os.path.exists(f):
        os.remove(f)

for name, record in sequences.items():
    filename = f"{name}.fasta"
    with open(filename, "w") as f:
        f.write(f">{name}|{record.id}|{record.description}\n")
        f.write(str(record.seq) + "\n")

for f in sorted(os.listdir(".")):
    if f.endswith(".fasta"):
        print(" -", f)

In [ ]:
if os.path.exists("NEAT1_v1.fasta"):
    os.remove("NEAT1_v1.fasta")

for f in sorted(os.listdir(".")):
    if f.endswith(".fasta"):
        print(" -", f)

In [ ]:
from Bio import Entrez, SeqIO

Entrez.email = "your_email@example.com"

# Accession used by Brown et al. (2012) for MENb numbering
handle = Entrez.efetch(db="nucleotide", id="GQ859162.1", rettype="fasta", retmode="text")
record_correct = SeqIO.read(handle, "fasta")
handle.close()

print(record_correct.id)
print(record_correct.description)
print(len(record_correct.seq))

In [ ]:
start = 22643 - 1
end = 22743

ene_a_domain = record_correct.seq[start:end]

with open("NEAT1_ENE_A_domain.fasta", "w") as f:
    f.write(">NEAT1_MENbeta_ENE_A_rich_tract|GQ859162.1|22643-22743\n")
    f.write(str(ene_a_domain) + "\n")

In [ ]:
!pip install ViennaRNA

## 2. RNA secondary structure (ViennaRNA)

In [ ]:
import RNA

with open("NEAT1_ENE_A_domain.fasta") as f:
    lines = f.readlines()
    header = lines[0].strip()
    seq_dna = lines[1].strip()

seq_rna = seq_dna.replace("T", "U")

fc = RNA.fold_compound(seq_rna)
structure, mfe = fc.mfe()

print(seq_rna)
print(structure)

with open("NEAT1_ENE_A_secondary_structure.txt", "w") as f:
    f.write(f"{header}\n")
    f.write(f"{seq_rna}\n")
    f.write(f"{structure}  ({mfe:.2f} kcal/mol)\n")

In [ ]:
constraint = "." * 81 + "x" * 20

print(constraint)

fc_constrained = RNA.fold_compound(seq_rna)
fc_constrained.hc_add_from_db(constraint)
structure_constrained, mfe_constrained = fc_constrained.mfe()

print(seq_rna)
print(structure_constrained)

In [ ]:
constraint_v2 = list("." * 101)
for i in range(13, 23):
    constraint_v2[i] = "x"
for i in range(81, 101):
    constraint_v2[i] = "x"
constraint_v2 = "".join(constraint_v2)

print(seq_rna)
print(constraint_v2)

fc_v2 = RNA.fold_compound(seq_rna)
fc_v2.hc_add_from_db(constraint_v2)
structure_v2, mfe_v2 = fc_v2.mfe()

print(seq_rna)
print(structure_v2)

u_rich_free = all(c == "." for c in structure_v2[13:23])
a_rich_free = all(c == "." for c in structure_v2[81:101])

In [ ]:
RNA.svg_rna_plot(seq_rna, structure, "NEAT1_ENE_A_MFE_unconstrained.svg")

RNA.svg_rna_plot(seq_rna, structure_v2, "NEAT1_ENE_A_MFE_constrained.svg")

print(" - NEAT1_ENE_A_MFE_unconstrained.svg")
print(" - NEAT1_ENE_A_MFE_constrained.svg")

from IPython.display import SVG, display
display(SVG("NEAT1_ENE_A_MFE_unconstrained.svg"))

In [ ]:
display(SVG("NEAT1_ENE_A_MFE_constrained.svg"))

In [ ]:
macro_colors = []
for i in range(101):
    if 13 <= i <= 22:
        macro_colors.append((i, i+1, 0.3, 0.8, 0.3))
    elif 81 <= i <= 100:
        macro_colors.append((i, i+1, 0.6, 0.3, 0.8))
    else:
        macro_colors.append((i, i+1, 0.7, 0.7, 0.7))

RNA.svg_rna_plot(
    seq_rna,
    structure_v2,
    "NEAT1_ENE_A_annotated.svg",
    macro_colors
)

display(SVG("NEAT1_ENE_A_annotated.svg"))

In [ ]:
with open("for_rnaplot.txt", "w") as f:
    f.write(seq_rna + "\n")
    f.write(structure_v2 + "\n")

pre_annotation = (
    "14 23 9 GREEN Fomark "
    "82 101 9 BLUE Fomark "
)

!echo "{seq_rna}" > rnaplot_input.txt
!echo "{structure_v2}" >> rnaplot_input.txt
!cat rnaplot_input.txt | RNAplot --pre "{pre_annotation}" -o svg

import os
for f in os.listdir("."):
    if f.endswith(".svg"):
        print(" -", f)

In [ ]:
with open("README.md") as f:
    print(f.read())

In [ ]:
!pip install torch einops biopython omegaconf ml-collections --break-system-packages -q
!pip install -e . --break-system-packages -q

In [ ]:
import os

os.makedirs("example/input/NEAT1_ENE_A", exist_ok=True)

with open("example/input/NEAT1_ENE_A/NEAT1_ENE_A.fasta", "w") as f:
    f.write(">NEAT1_MENbeta_ENE_A_rich_tract|GQ859162.1|22643-22743\n")
    f.write("GCCGCCGCAGGUGUUUCUUUUACUGAGUGCAGCCCAUGGCCGCACUCAGGUUUUGCUUUUCACCUUCCCAUCUGUGAAAGAGUGAGCAGGAAAAAGCAAAA\n")

with open("example/input/NEAT1_ENE_A/NEAT1_ENE_A.fasta") as f:
    print(f.read())

In [ ]:
!pip install openmm pdbfixer --break-system-packages -q

## 3. 3D structure prediction (RhoFold+)

In [ ]:
!python inference.py \
  --input_fas example/input/NEAT1_ENE_A/NEAT1_ENE_A.fasta \
  --single_seq_pred True \
  --output_dir example/output/NEAT1_ENE_A/ \
  --ckpt pretrained/RhoFold_pretrained.pt \
  --device cuda:0 \
  --relax_steps 0

In [ ]:
import os

output_dir = "example/output/NEAT1_ENE_A/"
for f in sorted(os.listdir(output_dir)):
    filepath = os.path.join(output_dir, f)
    size_kb = os.path.getsize(filepath) / 1024
    print(f" - {f}  ({size_kb:.1f} KB)")

with open(os.path.join(output_dir, "unrelaxed_model.pdb")) as f:
    for i, line in enumerate(f):
        if i >= 15:
            break
        print(line.rstrip())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil, os

project_dir = "/content/drive/MyDrive/NEAT1_project"
os.makedirs(project_dir, exist_ok=True)

files_to_save = [
    "/content/NEAT1_ENE_A_RhoFold.pdb",
    "/content/RhoFold/example/output/NEAT1_ENE_A/ss.ct",
    "/content/RhoFold/example/output/NEAT1_ENE_A/results.npz",
    "/content/RhoFold/example/output/NEAT1_ENE_A/log.txt",
]
for fpath in files_to_save:
    if os.path.exists(fpath):
        shutil.copy(fpath, project_dir)

In [ ]:
import os

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

trrosetta_dir = "/content/drive/MyDrive/NEAT1_project/trRosettaRNA_results"
os.makedirs(trrosetta_dir, exist_ok=True)

!wget https://yanglab.qd.sdu.edu.cn/trRosettaRNA/output/RNA031693/RNA031693_results.tar.bz2 -O /content/RNA031693_results.tar.bz2

!tar -xjf /content/RNA031693_results.tar.bz2 -C {trrosetta_dir}

for root, dirs, files in os.walk(trrosetta_dir):
    for f in files:
        print(" -", os.path.join(root, f).replace(trrosetta_dir, ""))

## 4. Structural comparison (C1' RMSD, base pairs)

In [ ]:
!pip install py3Dmol --break-system-packages -q

In [ ]:
import py3Dmol

rhofold_pdb_path = "/content/drive/MyDrive/NEAT1_project/NEAT1_ENE_A_RhoFold.pdb"
trrosetta_pdb_path = "/content/drive/MyDrive/NEAT1_project/trRosettaRNA_results/RNA031693_results/model1.pdb"

with open(rhofold_pdb_path) as f:
    rhofold_pdb_content = f.read()

with open(trrosetta_pdb_path) as f:
    trrosetta_pdb_content = f.read()

In [ ]:
view = py3Dmol.view(viewergrid=(1, 2), width=900, height=450)

view.addModel(rhofold_pdb_content, "pdb", viewer=(0, 0))
view.setStyle({"cartoon": {"color": "spectrum"}}, viewer=(0, 0))
view.zoomTo(viewer=(0, 0))

view.addModel(trrosetta_pdb_content, "pdb", viewer=(0, 1))
view.setStyle({"cartoon": {"color": "spectrum"}}, viewer=(0, 1))
view.zoomTo(viewer=(0, 1))

view.show()

In [ ]:
!pip install barnaba --break-system-packages -q

In [ ]:
import barnaba as bb

stackings_rf, pairings_rf, res_rf = bb.annotate(rhofold_pdb_path)

stackings_tr, pairings_tr, res_tr = bb.annotate(trrosetta_pdb_path)

## 5. Homologs (BLAST) and multiple sequence alignment (MUSCLE)

In [ ]:
from Bio.Blast import NCBIWWW, NCBIXML

query_seq = "GCCGCCGCAGGTGTTTCTTTTACTGAGTGCAGCCCATGGCCGCACTCAGGTTTTGCTTTTCACCTTCCCATCTGTGAAAGAGTGAGCAGGAAAAAGCAAAA"

result_handle = NCBIWWW.qblast(
    program="blastn",
    database="nt",
    sequence=query_seq,
    entrez_query="Mammalia[Organism]",
    hitlist_size=50,
    expect=10
)

with open("/content/drive/MyDrive/NEAT1_project/blast_results_raw.xml", "w") as f:
    f.write(result_handle.read())

In [ ]:
!pip install biopython --break-system-packages -q

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

In [ ]:
from Bio.Blast import NCBIXML

with open("/content/drive/MyDrive/NEAT1_project/blast_results_raw.xml") as f:
    blast_record = NCBIXML.read(f)

query_seq = "GCCGCCGCAGGTGTTTCTTTTACTGAGTGCAGCCCATGGCCGCACTCAGGTTTTGCTTTTCACCTTCCCATCTGTGAAAGAGTGAGCAGGAAAAAGCAAAA"

hits_summary = []
for i, alignment in enumerate(blast_record.alignments[:10]):
    hsp = alignment.hsps[0]
    identity_pct = (hsp.identities / hsp.align_length) * 100
    print(f"{i+1}. {alignment.title[:80]}")
    print(f"   Length: {alignment.length}, E-value: {hsp.expect:.2e}, Identity: {identity_pct:.1f}%, Query coverage: {hsp.align_length}/{len(query_seq)}")
    print(f"   Query:  {hsp.query[:60]}...")
    print(f"   Match:  {hsp.match[:60]}...")
    print(f"   Sbjct:  {hsp.sbjct[:60]}...")
    print()
    hits_summary.append({
        "title": alignment.title,
        "length": alignment.length,
        "evalue": hsp.expect,
        "identity_pct": identity_pct,
        "sbjct_seq": hsp.sbjct,
        "accession": alignment.accession
    })

In [ ]:
species_seen = set()
for i, alignment in enumerate(blast_record.alignments):
    hsp = alignment.hsps[0]
    identity_pct = (hsp.identities / hsp.align_length) * 100
    title_short = alignment.title[:70]
    print(f"{i+1}. Identity: {identity_pct:.1f}%  | {title_short}")

non_human_hits = [h for h in hits_summary if h['identity_pct'] < 100]

In [ ]:
seen_species = set()
selected_records = []

for i, alignment in enumerate(blast_record.alignments):
    hsp = alignment.hsps[0]
    identity_pct = (hsp.identities / hsp.align_length) * 100
    title = alignment.title

    import re
    match = re.search(r'(?:PREDICTED:\s*)?([A-Z][a-z]+ [a-z]+)', title)
    species = match.group(1) if match else f"unknown_{i}"

    if species not in seen_species:
        seen_species.add(species)
        selected_records.append({
            "species": species,
            "accession": alignment.accession,
            "identity_pct": identity_pct,
            "sbjct_aligned": hsp.sbjct,
            "query_aligned": hsp.query
        })

for r in selected_records:
    print(f"{r['species']:30s} | {r['accession']:15s} | Identity: {r['identity_pct']:.1f}%")

In [ ]:
with open("/content/drive/MyDrive/NEAT1_project/homologs_raw.fasta", "w") as f:
    f.write(f">Homo_sapiens_query\n{query_seq}\n")

    for r in selected_records:
        if r['species'] == "Homo sapiens":
            continue
        species_clean = r['species'].replace(" ", "_")
        seq_no_gap = r['sbjct_aligned'].replace("-", "")
        f.write(f">{species_clean}_{r['accession']}\n{seq_no_gap}\n")

with open("/content/drive/MyDrive/NEAT1_project/homologs_raw.fasta") as f:
    content = f.read()
    print(content[:500])

In [ ]:
!apt-get install -y muscle -q 2>&1 | tail -5

In [ ]:
!muscle -in /content/drive/MyDrive/NEAT1_project/homologs_raw.fasta -out /content/drive/MyDrive/NEAT1_project/homologs_aligned.fasta

In [ ]:
from Bio import AlignIO

alignment = AlignIO.read("/content/drive/MyDrive/NEAT1_project/homologs_aligned.fasta", "fasta")

print()

for record in alignment:
    print(f"{record.id:35s} {str(record.seq)}")

In [ ]:
from Bio import AlignIO

alignment = AlignIO.read("/content/drive/MyDrive/NEAT1_project/homologs_aligned.fasta", "fasta")

query_record = None
other_records = []
for record in alignment:
    if record.id == "Homo_sapiens_query":
        query_record = record
    else:
        other_records.append(record)

assert query_record is not None
assert "-" not in str(query_record.seq)

a3m_path = "/content/drive/MyDrive/NEAT1_project/NEAT1_ENE_A.a3m"
with open(a3m_path, "w") as f:
    f.write(f">{query_record.id}\n{str(query_record.seq)}\n")
    for record in other_records:
        f.write(f">{record.id}\n{str(record.seq)}\n")

with open(a3m_path) as f:
    content = f.read()
    print(content[:300])

In [ ]:
import torch

In [ ]:
import os

if os.path.exists("/content/RhoFold"):
    %cd /content/RhoFold
else:
    %cd /content
    !git clone https://github.com/ml4bio/RhoFold.git
    %cd RhoFold
    !pip install torch einops biopython omegaconf ml-collections openmm pdbfixer --break-system-packages -q
    !pip install -e . --break-system-packages -q

    os.makedirs("pretrained", exist_ok=True)
    !wget https://huggingface.co/cuhkaih/rhofold/resolve/main/rhofold_pretrained_params.pt -O pretrained/RhoFold_pretrained.pt -q

!ls -la

In [ ]:
import os, shutil

os.makedirs("example/input/NEAT1_ENE_A_MSA", exist_ok=True)

with open("example/input/NEAT1_ENE_A_MSA/NEAT1_ENE_A.fasta", "w") as f:
    f.write(">NEAT1_MENbeta_ENE_A_rich_tract|GQ859162.1|22643-22743\n")
    f.write("GCCGCCGCAGGUGUUUCUUUUACUGAGUGCAGCCCAUGGCCGCACUCAGGUUUUGCUUUUCACCUUCCCAUCUGUGAAAGAGUGAGCAGGAAAAAGCAAAA\n")

shutil.copy(
    "/content/drive/MyDrive/NEAT1_project/NEAT1_ENE_A.a3m",
    "example/input/NEAT1_ENE_A_MSA/NEAT1_ENE_A.a3m"
)

!ls -la example/input/NEAT1_ENE_A_MSA/

In [ ]:
!python inference.py \
  --input_fas example/input/NEAT1_ENE_A_MSA/NEAT1_ENE_A.fasta \
  --input_a3m example/input/NEAT1_ENE_A_MSA/NEAT1_ENE_A.a3m \
  --output_dir example/output/NEAT1_ENE_A_MSA/ \
  --ckpt pretrained/RhoFold_pretrained.pt \
  --device cuda:0 \
  --relax_steps 0

In [ ]:
import shutil

shutil.copy(
    "example/output/NEAT1_ENE_A_MSA/unrelaxed_model.pdb",
    "/content/drive/MyDrive/NEAT1_project/NEAT1_ENE_A_RhoFold_withMSA.pdb"
)

plddt_scores = []
with open("example/output/NEAT1_ENE_A_MSA/unrelaxed_model.pdb") as f:
    for line in f:
        if line.startswith("ATOM"):
            bfactor = float(line[60:66].strip())
            plddt_scores.append(bfactor)

import numpy as np

In [ ]:
from Bio.PDB import PDBParser, Superimposer

parser = PDBParser(QUIET=True)

structure_no_msa = parser.get_structure("RhoFold_noMSA", "/content/drive/MyDrive/NEAT1_project/NEAT1_ENE_A_RhoFold.pdb")
structure_with_msa = parser.get_structure("RhoFold_MSA", "/content/drive/MyDrive/NEAT1_project/NEAT1_ENE_A_RhoFold_withMSA.pdb")

def get_c1_atoms(structure):
    atoms = []
    for model in structure:
        for chain in model:
            for residue in chain:
                if "C1'" in residue:
                    atoms.append(residue["C1'"])
        break
    return atoms

atoms_no_msa = get_c1_atoms(structure_no_msa)
atoms_with_msa = get_c1_atoms(structure_with_msa)

sup = Superimposer()
sup.set_atoms(atoms_no_msa, atoms_with_msa)
sup.apply(atoms_with_msa)

structure_trrosetta = parser.get_structure("trRosetta", "/content/drive/MyDrive/NEAT1_project/trRosettaRNA_results/RNA031693_results/model1.pdb")
atoms_trrosetta = get_c1_atoms(structure_trrosetta)

sup2 = Superimposer()
sup2.set_atoms(atoms_trrosetta, atoms_with_msa)
sup2.apply(atoms_with_msa)

In [ ]:
from Bio.PDB import PDBParser, Superimposer

parser2 = PDBParser(QUIET=True)

def get_c1_coords(pdb_path):
    struct = parser2.get_structure("s", pdb_path)
    atoms = []
    for model in struct:
        for chain in model:
            for residue in chain:
                if "C1'" in residue:
                    atoms.append(residue["C1'"])
        break
    return atoms

path_no_msa = "/content/drive/MyDrive/NEAT1_project/NEAT1_ENE_A_RhoFold.pdb"
path_with_msa = "/content/drive/MyDrive/NEAT1_project/NEAT1_ENE_A_RhoFold_withMSA.pdb"
path_trrosetta = "/content/drive/MyDrive/NEAT1_project/trRosettaRNA_results/RNA031693_results/model1.pdb"

a1 = get_c1_coords(path_no_msa)
a2 = get_c1_coords(path_with_msa)
s1 = Superimposer()
s1.set_atoms(a1, a2)
print(f"RMSD (RhoFold no-MSA vs with-MSA): {s1.rms:.2f} Å")

a3 = get_c1_coords(path_no_msa)
a4 = get_c1_coords(path_trrosetta)
s2 = Superimposer()
s2.set_atoms(a3, a4)
print(f"RMSD (RhoFold no-MSA vs trRosettaRNA): {s2.rms:.2f} Å")

a5 = get_c1_coords(path_with_msa)
a6 = get_c1_coords(path_trrosetta)
s3 = Superimposer()
s3.set_atoms(a5, a6)
print(f"RMSD (RhoFold with-MSA vs trRosettaRNA): {s3.rms:.2f} Å")

In [ ]:
a_self1 = get_c1_coords(path_no_msa)
a_self2 = get_c1_coords(path_no_msa)
s_self = Superimposer()
s_self.set_atoms(a_self1, a_self2)

In [ ]:
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
!bash miniconda.sh -b -p /usr/local/miniconda
!/usr/local/miniconda/bin/conda init bash

## 6. Pocket detection (fpocketR)

In [ ]:
fpocketR_python = "/usr/local/miniconda/envs/fpocketR/bin/python"

!{fpocketR_python} -m fpocketR --help

In [ ]:
import os
os.environ["PATH"] = "/usr/local/miniconda/bin:" + os.environ["PATH"]

!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

!conda config --add channels defaults
!conda config --add channels bioconda
!conda config --add channels conda-forge

!conda create -n fpocketR fpocketr -y

In [ ]:
import os
env_fix = os.environ.copy()
env_fix["MPLBACKEND"] = "Agg"

import subprocess
result = subprocess.run(
    ["/usr/local/miniconda/envs/fpocketR/bin/python", "-m", "fpocketR", "--help"],
    env=env_fix,
    capture_output=True,
    text=True
)
print("STDOUT:")
print(result.stdout)
print("\nSTDERR:")
print(result.stderr)

In [ ]:
import os, subprocess

env_fix = os.environ.copy()
env_fix["MPLBACKEND"] = "Agg"

output_dir = "/content/drive/MyDrive/NEAT1_project/fpocketR_results"
os.makedirs(output_dir, exist_ok=True)

pdb_path = "/content/drive/MyDrive/NEAT1_project/NEAT1_ENE_A_RhoFold.pdb"

result = subprocess.run(
    [
        "/usr/local/miniconda/envs/fpocketR/bin/python", "-m", "fpocketR",
        "-pdb", pdb_path,
        "-o", output_dir,
        "-n", "RhoFold_noMSA",
        "-y"
    ],
    env=env_fix,
    capture_output=True,
    text=True
)

print("STDOUT:")
print(result.stdout)
print("\nSTDERR:")
print(result.stderr)
print(f"\nReturn code: {result.returncode}")

In [ ]:
result_dir = os.path.join(output_dir, "RhoFold_noMSA_clean_out")
for root, dirs, files in os.walk(result_dir):
    level = root.replace(result_dir, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        filepath = os.path.join(root, f)
        size_kb = os.path.getsize(filepath) / 1024
        print(f"{indent}  {f} ({size_kb:.1f} KB)")

In [ ]:
info_path = os.path.join(result_dir, "RhoFold_noMSA_clean_info.txt")
with open(info_path) as f:
    content = f.read()
print(content)

In [ ]:
def get_pocket_residues(pocket_pdb_path):
    residues = set()
    with open(pocket_pdb_path) as f:
        for line in f:
            if line.startswith("ATOM") or line.startswith("HETATM"):
                res_id = line[22:26].strip()
                residues.add(int(res_id))
    return sorted(residues)

pockets_dir = os.path.join(result_dir, "pockets")
for i in range(1, 7):
    pocket_file = os.path.join(pockets_dir, f"pocket{i}_atm.pdb")
    residues = get_pocket_residues(pocket_file)

In [ ]:
import os, subprocess

env_fix = os.environ.copy()
env_fix["MPLBACKEND"] = "Agg"

pdb_path_tr = "/content/drive/MyDrive/NEAT1_project/trRosettaRNA_results/RNA031693_results/model1.pdb"

result_tr = subprocess.run(
    [
        "/usr/local/miniconda/envs/fpocketR/bin/python", "-m", "fpocketR",
        "-pdb", pdb_path_tr,
        "-o", output_dir,
        "-n", "trRosettaRNA",
        "-y"
    ],
    env=env_fix,
    capture_output=True,
    text=True
)

print("STDOUT:")
print(result_tr.stdout)
print("\nSTDERR:")
print(result_tr.stderr)
print(f"\nReturn code: {result_tr.returncode}")

In [ ]:
import os, subprocess

env_fix = os.environ.copy()
env_fix["MPLBACKEND"] = "Agg"

pdb_path_msa = "/content/drive/MyDrive/NEAT1_project/NEAT1_ENE_A_RhoFold_withMSA.pdb"

result_msa = subprocess.run(
    [
        "/usr/local/miniconda/envs/fpocketR/bin/python", "-m", "fpocketR",
        "-pdb", pdb_path_msa,
        "-o", output_dir,
        "-n", "RhoFold_withMSA",
        "-y"
    ],
    env=env_fix,
    capture_output=True,
    text=True
)

print("STDOUT:")
print(result_msa.stdout)
print("\nSTDERR:")
print(f"\nReturn code: {result_msa.returncode}")

In [ ]:
result_dir_msa = os.path.join(output_dir, "RhoFold_withMSA_clean_out")

for root, dirs, files in os.walk(result_dir_msa):
    level = root.replace(result_dir_msa, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        filepath = os.path.join(root, f)
        size_kb = os.path.getsize(filepath) / 1024
        print(f"{indent}  {f} ({size_kb:.1f} KB)")

info_path_msa = os.path.join(result_dir_msa, "RhoFold_withMSA_clean_info.txt")
with open(info_path_msa) as f:
    print(f.read())

In [ ]:
pockets_dir_msa = os.path.join(result_dir_msa, "pockets")
pocket1_msa_residues = get_pocket_residues(os.path.join(pockets_dir_msa, "pocket1_atm.pdb"))

u_rich = set(range(14, 24))
a_rich = set(range(82, 102))
pocket_set = set(pocket1_msa_residues)

In [ ]:
import subprocess

result = subprocess.run(
    ["/usr/local/miniconda/envs/gromacs/bin/gmx", "--version"],
    capture_output=True,
    text=True
)
print("STDOUT:")
print(result.stdout)
print("\nSTDERR:")
print(result.stderr)

In [ ]:
import os

md_dir = "/content/drive/MyDrive/NEAT1_project/MD_simulation"
os.makedirs(md_dir, exist_ok=True)

import shutil
shutil.copy(
    "/content/drive/MyDrive/NEAT1_project/NEAT1_ENE_A_RhoFold_withMSA.pdb",
    os.path.join(md_dir, "start.pdb")
)

with open(os.path.join(md_dir, "start.pdb")) as f:
    lines = f.readlines()

print("".join(lines[:5]))
print("".join(lines[-5:]))

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, shutil

md_dir = "/content/drive/MyDrive/NEAT1_project/MD_simulation"
os.makedirs(md_dir, exist_ok=True)

source_pdb = "/content/drive/MyDrive/NEAT1_project/NEAT1_ENE_A_RhoFold_withMSA.pdb"
start_pdb = os.path.join(md_dir, "start.pdb")
shutil.copy(source_pdb, start_pdb)

with open(start_pdb) as f:
    lines = f.readlines()

cleaned_lines2 = []
for line in lines:
    if line.startswith("MODEL") or line.startswith("ENDMDL"):
        continue
    if line.startswith("ATOM") or line.startswith("HETATM"):
        tail = line[76:].rstrip("\n")
        clean_element = "".join(c for c in tail if c.isalpha())[:2]
        new_line = line[:76] + f"{clean_element:>2}\n"
        cleaned_lines2.append(new_line)
    else:
        cleaned_lines2.append(line)

cleaned_path2 = os.path.join(md_dir, "start_clean.pdb")
with open(cleaned_path2, "w") as f:
    f.writelines(cleaned_lines2)

In [ ]:
cleaned_lines = []
for line in lines:
    if line.startswith("MODEL") or line.startswith("ENDMDL"):
        continue
    if line.startswith("ATOM") or line.startswith("HETATM"):
        element = line[76:78].strip()
        if len(element) > 2 or any(c in element for c in "+-0123456789"):
            clean_element = "".join(c for c in element if c.isalpha())
            line = line[:76] + f"{clean_element:>2}" + line[78:]
        cleaned_lines.append(line)
    else:
        cleaned_lines.append(line)

cleaned_path = os.path.join(md_dir, "start_clean.pdb")
with open(cleaned_path, "w") as f:
    f.writelines(cleaned_lines)

with open(cleaned_path) as f:
    check_lines = f.readlines()
print("".join(check_lines[-5:]))

## 7. MD system preparation (GROMACS)

In [ ]:
import subprocess

gmx_bin = "/usr/local/miniconda/envs/gromacs/bin.AVX2_256/gmx"

result = subprocess.run(
    [gmx_bin, "pdb2gmx", "-h"],
    capture_output=True,
    text=True
)
print(result.stdout[-2000:])
print(result.stderr[-2000:])

In [ ]:
import subprocess

gmx_bin = "/usr/local/miniconda/envs/gromacs/bin.AVX2_256/gmx"

result = subprocess.run(
    [gmx_bin, "pdb2gmx", "-h"],
    capture_output=True,
    text=True
)
output = result.stdout + result.stderr
print(output[-3000:])

In [ ]:
import os

ff_dir = "/usr/local/miniconda/envs/gromacs/share/gromacs/top"
for item in sorted(os.listdir(ff_dir)):
    if item.endswith(".ff"):
        print(" -", item)

In [ ]:
import subprocess

result = subprocess.run(
    ["git", "clone", "https://github.com/srnas/ff.git", "/content/srnas_ff"],
    capture_output=True,
    text=True
)
print(result.stdout)
print(result.stderr)

import os
for item in sorted(os.listdir("/content/srnas_ff")):
    print(" -", item)

In [ ]:
import shutil, os

src_ff = "/content/srnas_ff/amber_na.ff"
dst_ff = "/usr/local/miniconda/envs/gromacs/share/gromacs/top/amber_na.ff"

shutil.copytree(src_ff, dst_ff)

for item in sorted(os.listdir(dst_ff)):
    print(" -", item)

top_dir = "/usr/local/miniconda/envs/gromacs/share/gromacs/top"
for item in sorted(os.listdir(top_dir)):
    if item.endswith(".ff"):
        print(" -", item)

In [ ]:
import subprocess, os

gmx_bin = "/usr/local/miniconda/envs/gromacs/bin.AVX2_256/gmx"
md_dir = "/content/drive/MyDrive/NEAT1_project/MD_simulation"

os.chdir(md_dir)

result = subprocess.run(
    [
        gmx_bin, "pdb2gmx",
        "-f", "start_clean.pdb",
        "-o", "processed.gro",
        "-p", "topol.top",
        "-ff", "amber_na",
        "-water", "tip3p",
        "-ignh"
    ],
    capture_output=True,
    text=True
)

print("STDOUT:")
print(result.stdout[-3000:])
print("\nSTDERR:")
print(result.stderr[-3000:])
print(f"\nReturn code: {result.returncode}")

In [ ]:
with open("start_clean.pdb") as f:
    lines = f.readlines()

new_lines = []
removed_count = 0
for line in lines:
    if line.startswith("ATOM"):
        res_num = int(line[22:26].strip())
        atom_name = line[12:16].strip()
        if res_num == 1 and atom_name in ["P", "OP1", "OP2"]:
            removed_count += 1
            continue
    new_lines.append(line)

with open("start_clean_5trim.pdb", "w") as f:
    f.writelines(new_lines)

with open("start_clean_5trim.pdb") as f:
    check = f.readlines()
print("".join(check[:10]))

In [ ]:
result2 = subprocess.run(
    [
        gmx_bin, "pdb2gmx",
        "-f", "start_clean_5trim.pdb",
        "-o", "processed.gro",
        "-p", "topol.top",
        "-ff", "amber_na",
        "-water", "tip3p",
        "-ignh"
    ],
    capture_output=True,
    text=True
)

print("STDOUT:")
print(result2.stdout[-2000:])
print("\nSTDERR:")
print(result2.stderr[-2500:])
print(f"\nReturn code: {result2.returncode}")

In [ ]:
with open("processed.gro") as f:
    gro_lines = f.readlines()

for atom_idx in [2303, 2304, 2467, 2468]:
    line = gro_lines[atom_idx + 1]
    print(f"Atom {atom_idx}: {line.rstrip()}")

In [ ]:
import shutil, os

source_nomsa = "/content/drive/MyDrive/NEAT1_project/NEAT1_ENE_A_RhoFold.pdb"
shutil.copy(source_nomsa, "start_nomsa.pdb")

with open("start_nomsa.pdb") as f:
    lines_nomsa = f.readlines()

cleaned_nomsa = []
for line in lines_nomsa:
    if line.startswith("MODEL") or line.startswith("ENDMDL"):
        continue
    if line.startswith("ATOM") or line.startswith("HETATM"):
        tail = line[76:].rstrip("\n")
        clean_element = "".join(c for c in tail if c.isalpha())[:2]
        new_line = line[:76] + f"{clean_element:>2}\n"
        cleaned_nomsa.append(new_line)
    else:
        cleaned_nomsa.append(line)

final_nomsa = []
removed = 0
for line in cleaned_nomsa:
    if line.startswith("ATOM"):
        res_num = int(line[22:26].strip())
        atom_name = line[12:16].strip()
        if res_num == 1 and atom_name in ["P", "OP1", "OP2"]:
            removed += 1
            continue
    final_nomsa.append(line)

with open("start_nomsa_clean.pdb", "w") as f:
    f.writelines(final_nomsa)

os.makedirs("test_nomsa", exist_ok=True)
shutil.copy("start_nomsa_clean.pdb", "test_nomsa/start_nomsa_clean.pdb")
os.chdir("test_nomsa")

result_nomsa = subprocess.run(
    [
        gmx_bin, "pdb2gmx",
        "-f", "start_nomsa_clean.pdb",
        "-o", "processed.gro",
        "-p", "topol.top",
        "-ff", "amber_na",
        "-water", "tip3p",
        "-ignh"
    ],
    capture_output=True,
    text=True
)

os.chdir(md_dir)

long_bonds = [l for l in result_nomsa.stdout.split("\n") if "Long Bond" in l]
for lb in long_bonds:
    print(" ", lb)

print(f"\nReturn code: {result_nomsa.returncode}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
!bash miniconda.sh -b -p /usr/local/miniconda

import os
os.environ["PATH"] = "/usr/local/miniconda/bin:" + os.environ["PATH"]

!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

!conda create -n gromacs -c bioconda -c conda-forge gromacs -y

import subprocess
subprocess.run(["git", "clone", "https://github.com/srnas/ff.git", "/content/srnas_ff"])

import shutil
dst_ff = "/usr/local/miniconda/envs/gromacs/share/gromacs/top/amber_na.ff"
if not os.path.exists(dst_ff):
    shutil.copytree("/content/srnas_ff/amber_na.ff", dst_ff)

In [ ]:
import os, shutil, subprocess

gmx_bin = "/usr/local/miniconda/envs/gromacs/bin.AVX2_256/gmx"
md_dir = "/content/drive/MyDrive/NEAT1_project/MD_simulation"
os.chdir(md_dir)

for f in sorted(os.listdir(md_dir)):
    print(" -", f)

In [ ]:
os.makedirs("test_nomsa", exist_ok=True)
shutil.copy("start_nomsa_clean.pdb", "test_nomsa/start_nomsa_clean.pdb")
os.chdir("test_nomsa")

result_nomsa = subprocess.run(
    [
        gmx_bin, "pdb2gmx",
        "-f", "start_nomsa_clean.pdb",
        "-o", "processed.gro",
        "-p", "topol.top",
        "-ff", "amber_na",
        "-water", "tip3p",
        "-ignh"
    ],
    capture_output=True,
    text=True
)

os.chdir(md_dir)

long_bonds_nomsa = [l for l in result_nomsa.stdout.split("\n") if "Long Bond" in l]
for lb in long_bonds_nomsa:
    print(" ", lb)

print(f"\nReturn code: {result_nomsa.returncode}")

if result_nomsa.returncode != 0:
    print(result_nomsa.stderr[-1500:])

In [ ]:
os.chdir("test_nomsa")
with open("processed.gro") as f:
    gro_lines_nomsa = f.readlines()

for atom_idx in [255, 256]:
    line = gro_lines_nomsa[atom_idx + 1]
    print(f"Atom {atom_idx}: {line.rstrip()}")
os.chdir(md_dir)

In [ ]:
import os
os.chdir(os.path.join(md_dir, "test_nomsa"))

result_box = subprocess.run(
    [
        gmx_bin, "editconf",
        "-f", "processed.gro",
        "-o", "boxed.gro",
        "-c",
        "-d", "1.2",
        "-bt", "cubic"
    ],
    capture_output=True,
    text=True
)

print("STDOUT:")
print(result_box.stdout[-1500:])
print("\nSTDERR:")
print(result_box.stderr[-1500:])
print(f"\nReturn code: {result_box.returncode}")

In [ ]:
result_box2 = subprocess.run(
    [
        gmx_bin, "editconf",
        "-f", "processed.gro",
        "-o", "boxed.gro",
        "-c",
        "-d", "0.9",
        "-bt", "cubic"
    ],
    capture_output=True,
    text=True
)

print("STDOUT:")
print(result_box2.stdout[-1200:])
print(f"\nReturn code: {result_box2.returncode}")

In [ ]:
result_box3 = subprocess.run(
    [
        gmx_bin, "editconf",
        "-f", "processed.gro",
        "-o", "boxed.gro",
        "-c",
        "-d", "0.9",
        "-bt", "dodecahedron"
    ],
    capture_output=True,
    text=True
)

print("STDOUT:")
print(result_box3.stdout[-1200:])
print(f"\nReturn code: {result_box3.returncode}")

In [ ]:
result_solv = subprocess.run(
    [
        gmx_bin, "solvate",
        "-cp", "boxed.gro",
        "-cs", "spc216.gro",
        "-o", "solvated.gro",
        "-p", "topol.top"
    ],
    capture_output=True,
    text=True
)

print("STDOUT:")
print(result_solv.stdout[-2000:])
print("\nSTDERR:")
print(result_solv.stderr[-1500:])
print(f"\nReturn code: {result_solv.returncode}")

In [ ]:
with open("ions.mdp", "w") as f:
    f.write("integrator = steep\n")
    f.write("emtol = 1000.0\n")
    f.write("nsteps = 0\n")

result_grompp_ion = subprocess.run(
    [
        gmx_bin, "grompp",
        "-f", "ions.mdp",
        "-c", "solvated.gro",
        "-p", "topol.top",
        "-o", "ions.tpr",
        "-maxwarn", "2"
    ],
    capture_output=True,
    text=True
)

print("STDOUT:")
print(result_grompp_ion.stdout[-2000:])
print("\nSTDERR:")
print(result_grompp_ion.stderr[-2000:])
print(f"\nReturn code: {result_grompp_ion.returncode}")

In [ ]:
result_genion = subprocess.run(
    [
        gmx_bin, "genion",
        "-s", "ions.tpr",
        "-o", "solvated_ions.gro",
        "-p", "topol.top",
        "-pname", "NA",
        "-nname", "CL",
        "-neutral",
        "-conc", "0.15"
    ],
    input="SOL\n",
    capture_output=True,
    text=True
)

print("STDOUT:")
print(result_genion.stdout[-2000:])
print("\nSTDERR:")
print(result_genion.stderr[-1500:])
print(f"\nReturn code: {result_genion.returncode}")

In [ ]:
em_mdp_content = """
integrator      = steep
emtol           = 1000.0
emstep          = 0.01
nsteps          = 5000

cutoff-scheme   = Verlet
nstlist         = 1
ns_type         = grid
coulombtype     = PME
rcoulomb        = 1.0
rvdw            = 1.0
pbc             = xyz
"""

with open("em.mdp", "w") as f:
    f.write(em_mdp_content)

print(em_mdp_content)

In [ ]:
result_grompp_em = subprocess.run(
    [
        gmx_bin, "grompp",
        "-f", "em.mdp",
        "-c", "solvated_ions.gro",
        "-p", "topol.top",
        "-o", "em.tpr",
        "-maxwarn", "2"
    ],
    capture_output=True,
    text=True
)

print("STDOUT:")
print(result_grompp_em.stdout[-2000:])
print("\nSTDERR:")
print(result_grompp_em.stderr[-2000:])
print(f"\nReturn code: {result_grompp_em.returncode}")

In [ ]:
import time

start_time = time.time()

result_mdrun_em = subprocess.run(
    [
        gmx_bin, "mdrun",
        "-deffnm", "em",
        "-nt", "2"
    ],
    capture_output=True,
    text=True
)

elapsed = time.time() - start_time

print(result_mdrun_em.stdout[-2000:])
print(result_mdrun_em.stderr[-3000:])
print(f"\nReturn code: {result_mdrun_em.returncode}")

In [ ]:
nvt_mdp_content = """; NVT equilibration
define          = -DPOSRES
integrator      = md
nsteps          = 25000        ; 25000 x 2fs = 50 ps
dt              = 0.002

nstxout         = 500
nstvout         = 500
nstenergy       = 500
nstlog          = 500

continuation    = no
constraint_algorithm = lincs
constraints     = h-bonds

; neighbor searching
cutoff-scheme   = Verlet
nstlist         = 20
rcoulomb        = 1.0
rvdw            = 1.0
coulombtype     = PME

tcoupl          = V-rescale
tc-grps         = RNA Water_and_ions
tau_t           = 0.1     0.1
ref_t           = 300     300

pcoupl          = no

pbc             = xyz

gen_vel         = yes
gen_temp        = 300
gen_seed        = -1
"""

with open("nvt.mdp", "w") as f:
    f.write(nvt_mdp_content)

In [ ]:
result_ndx_check = subprocess.run(
    [gmx_bin, "make_ndx", "-f", "solvated_ions.gro", "-o", "index.ndx"],
    input="q\n",
    capture_output=True,
    text=True
)
print(result_ndx_check.stdout[-2500:])

In [ ]:
result_posre = subprocess.run(
    [gmx_bin, "genrestr", "-f", "processed.gro", "-o", "posre.itp"],
    input="1\n",
    capture_output=True,
    text=True
)
print("STDOUT:")
print(result_posre.stdout[-1500:])
print("\nSTDERR:")
print(result_posre.stderr[-800:])
print(f"\nReturn code: {result_posre.returncode}")

In [ ]:
result_grompp_nvt = subprocess.run(
    [
        gmx_bin, "grompp",
        "-f", "nvt.mdp",
        "-c", "em.gro",
        "-r", "em.gro",
        "-p", "topol.top",
        "-n", "index.ndx",
        "-o", "nvt.tpr",
        "-maxwarn", "2"
    ],
    capture_output=True,
    text=True
)

print("STDOUT:")
print(result_grompp_nvt.stdout[-2000:])
print("\nSTDERR:")
print(result_grompp_nvt.stderr[-2000:])
print(f"\nReturn code: {result_grompp_nvt.returncode}")

In [ ]:
import time

start_time = time.time()

result_mdrun_nvt = subprocess.run(
    [
        gmx_bin, "mdrun",
        "-deffnm", "nvt",
        "-nt", "2"
    ],
    capture_output=True,
    text=True
)

elapsed = time.time() - start_time

print(result_mdrun_nvt.stdout[-1500:])
print(result_mdrun_nvt.stderr[-3000:])
print(f"\nReturn code: {result_mdrun_nvt.returncode}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os
md_dir = "/content/drive/MyDrive/NEAT1_project/MD_simulation"
test_dir = os.path.join(md_dir, "test_nomsa")
os.chdir(test_dir)

for f in sorted(os.listdir(".")):
    if f.startswith("nvt"):
        size_mb = os.path.getsize(f) / (1024*1024)
        print(f" - {f} ({size_mb:.1f} MB)")

In [ ]:
import subprocess

gmx_bin = "/usr/local/miniconda/envs/gromacs/bin.AVX2_256/gmx"

result_temp = subprocess.run(
    [gmx_bin, "energy", "-f", "nvt.edr", "-o", "temperature.xvg"],
    input="Temperature\n0\n",
    capture_output=True,
    text=True
)

print("STDOUT:")
print(result_temp.stdout[-2000:])
print("\nSTDERR:")
print(result_temp.stderr[-500:])

In [ ]:
import os, subprocess

gmx_bin = "/usr/local/miniconda/envs/gromacs/bin.AVX2_256/gmx"

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

test_dir = "/content/drive/MyDrive/NEAT1_project/MD_simulation/test_nomsa"
os.chdir(test_dir)

In [ ]:
import subprocess

result_temp = subprocess.run(
    [gmx_bin, "energy", "-f", "nvt.edr", "-o", "temperature.xvg"],
    input="Temperature\n0\n",
    capture_output=True,
    text=True
)

print("STDOUT:")
print(result_temp.stdout[-2000:])
print("\nSTDERR:")
print(result_temp.stderr[-500:])
print(f"\nReturn code: {result_temp.returncode}")

In [ ]:
npt_mdp_content = """; NPT equilibration
define          = -DPOSRES
integrator      = md
nsteps          = 25000        ; 25000 x 2fs = 50 ps
dt              = 0.002

nstxout         = 500
nstvout         = 500
nstenergy       = 500
nstlog          = 500

continuation    = yes
constraint_algorithm = lincs
constraints     = h-bonds

; neighbor searching
cutoff-scheme   = Verlet
nstlist         = 20
rcoulomb        = 1.0
rvdw            = 1.0
coulombtype     = PME

tcoupl          = V-rescale
tc-grps         = RNA Water_and_ions
tau_t           = 0.1     0.1
ref_t           = 300     300

pcoupl          = C-rescale
pcoupltype      = isotropic
tau_p           = 2.0
ref_p           = 1.0
compressibility = 4.5e-5

; PBC
pbc             = xyz

gen_vel         = no
"""

with open("npt.mdp", "w") as f:
    f.write(npt_mdp_content)

In [ ]:
result_grompp_npt = subprocess.run(
    [
        gmx_bin, "grompp",
        "-f", "npt.mdp",
        "-c", "nvt.gro",
        "-r", "nvt.gro",
        "-t", "nvt.cpt",
        "-p", "topol.top",
        "-n", "index.ndx",
        "-o", "npt.tpr",
        "-maxwarn", "2"
    ],
    capture_output=True,
    text=True
)

print("STDOUT:")
print(result_grompp_npt.stdout[-2000:])
print("\nSTDERR:")
print(result_grompp_npt.stderr[-2000:])
print(f"\nReturn code: {result_grompp_npt.returncode}")

In [ ]:
with open("npt.mdp", "a") as f:
    f.write("\nrefcoord_scaling = com\n")

result_grompp_npt2 = subprocess.run(
    [
        gmx_bin, "grompp",
        "-f", "npt.mdp",
        "-c", "nvt.gro",
        "-r", "nvt.gro",
        "-t", "nvt.cpt",
        "-p", "topol.top",
        "-n", "index.ndx",
        "-o", "npt.tpr",
        "-maxwarn", "2"
    ],
    capture_output=True,
    text=True
)

print(result_grompp_npt2.stderr[-800:])
print(f"\nReturn code: {result_grompp_npt2.returncode}")

In [ ]:
import time

start_time = time.time()

result_mdrun_npt = subprocess.run(
    [
        gmx_bin, "mdrun",
        "-deffnm", "npt",
        "-nt", "2"
    ],
    capture_output=True,
    text=True
)

elapsed = time.time() - start_time

print(result_mdrun_npt.stderr[-2500:])
print(f"\nReturn code: {result_mdrun_npt.returncode}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, shutil, subprocess

if not os.path.exists("/usr/local/miniconda"):
    !wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
    !bash miniconda.sh -b -p /usr/local/miniconda

os.environ["PATH"] = "/usr/local/miniconda/bin:" + os.environ["PATH"]

if not os.path.exists("/usr/local/miniconda/envs/gromacs"):
    !conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
    !conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
    !conda create -n gromacs -c bioconda -c conda-forge gromacs -y

dst_ff = "/usr/local/miniconda/envs/gromacs/share/gromacs/top/amber_na.ff"
if not os.path.exists(dst_ff):
    subprocess.run(["git", "clone", "https://github.com/srnas/ff.git", "/content/srnas_ff"])
    shutil.copytree("/content/srnas_ff/amber_na.ff", dst_ff)

gmx_bin = "/usr/local/miniconda/envs/gromacs/bin.AVX2_256/gmx"

test_dir = "/content/drive/MyDrive/NEAT1_project/MD_simulation/test_nomsa"
os.chdir(test_dir)

for f in sorted(os.listdir(".")):
    if f.startswith("npt"):
        size_mb = os.path.getsize(f) / (1024*1024)
        print(f" - {f} ({size_mb:.2f} MB)")

In [ ]:
with open("npt.log") as f:
    log_content = f.read()

import re
steps_found = re.findall(r"Step\s+Time\s*\n\s*(\d+)\s+([\d.]+)", log_content)
for s in steps_found[-5:]:
    print(f"  Step: {s[0]}, Time: {s[1]} ps")

print(log_content[-1500:])

In [ ]:
import time

start_time = time.time()

result_mdrun_npt_continue = subprocess.run(
    [
        gmx_bin, "mdrun",
        "-deffnm", "npt",
        "-cpi", "npt.cpt",
        "-nt", "2"
    ],
    capture_output=True,
    text=True
)

elapsed = time.time() - start_time

print(result_mdrun_npt_continue.stderr[-2000:])
print(f"\nReturn code: {result_mdrun_npt_continue.returncode}")

In [ ]:
import time

start_time = time.time()

result_npt_retry = subprocess.run(
    [
        gmx_bin, "mdrun",
        "-deffnm", "npt",
        "-cpi", "npt.cpt",
        "-noappend",
        "-nt", "2"
    ],
    capture_output=True,
    text=True
)

elapsed = time.time() - start_time

print(result_npt_retry.stderr[-2000:])
print(f"\nReturn code: {result_npt_retry.returncode}")

In [ ]:
import os

for f in sorted(os.listdir(".")):
    if "part" in f:
        print(" -", f)

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os

test_dir = "/content/drive/MyDrive/NEAT1_project/MD_simulation/test_nomsa"
os.chdir(test_dir)

for f in sorted(os.listdir(".")):
    size_mb = os.path.getsize(f) / (1024*1024)
    print(f" - {f} ({size_mb:.2f} MB)")

In [ ]:
import os

for f in sorted(os.listdir(".")):
    if f.startswith("npt"):
        size_mb = os.path.getsize(f) / (1024*1024)
        mtime = os.path.getmtime(f)
        import datetime
        mtime_str = datetime.datetime.fromtimestamp(mtime).strftime("%H:%M:%S")

In [ ]:
import os, subprocess, shutil

if not os.path.exists("/usr/local/miniconda"):
    !wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
    !bash miniconda.sh -b -p /usr/local/miniconda

os.environ["PATH"] = "/usr/local/miniconda/bin:" + os.environ["PATH"]

if not os.path.exists("/usr/local/miniconda/envs/gromacs"):
    !conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
    !conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
    !conda create -n gromacs -c bioconda -c conda-forge gromacs -y

gmx_bin = "/usr/local/miniconda/envs/gromacs/bin.AVX2_256/gmx"

result_final_energy = subprocess.run(
    [gmx_bin, "energy", "-f", "npt.part0002.edr", "-o", "npt_temp_pressure.xvg"],
    input="Temperature\nPressure\n0\n",
    capture_output=True,
    text=True
)

print("\n" + result_final_energy.stdout[-1500:])

## 8. Production MD and trajectory analysis

In [ ]:
production_mdp_content = """; Production MD (unrestrained)
integrator      = md
nsteps          = 50000        ; 50000 x 2fs = 100 ps
dt              = 0.002

nstxout         = 5000
nstvout         = 5000
nstenergy       = 1000
nstlog          = 1000
nstxout-compressed = 1000

continuation    = yes
constraint_algorithm = lincs
constraints     = h-bonds

; neighbor searching
cutoff-scheme   = Verlet
nstlist         = 20
rcoulomb        = 1.0
rvdw            = 1.0
coulombtype     = PME

tcoupl          = V-rescale
tc-grps         = RNA Water_and_ions
tau_t           = 0.1     0.1
ref_t           = 300     300

pcoupl          = C-rescale
pcoupltype      = isotropic
tau_p           = 2.0
ref_p           = 1.0
compressibility = 4.5e-5

; PBC
pbc             = xyz

gen_vel         = no
"""

with open("production.mdp", "w") as f:
    f.write(production_mdp_content)

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, subprocess, shutil

if not os.path.exists("/usr/local/miniconda"):
    !wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
    !bash miniconda.sh -b -p /usr/local/miniconda

os.environ["PATH"] = "/usr/local/miniconda/bin:" + os.environ["PATH"]

if not os.path.exists("/usr/local/miniconda/envs/gromacs"):
    !conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
    !conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
    !conda create -n gromacs -c bioconda -c conda-forge gromacs -y

dst_ff = "/usr/local/miniconda/envs/gromacs/share/gromacs/top/amber_na.ff"
if not os.path.exists(dst_ff):
    subprocess.run(["git", "clone", "https://github.com/srnas/ff.git", "/content/srnas_ff"])
    shutil.copytree("/content/srnas_ff/amber_na.ff", dst_ff)

gmx_bin = "/usr/local/miniconda/envs/gromacs/bin.AVX2_256/gmx"

test_dir = "/content/drive/MyDrive/NEAT1_project/MD_simulation/test_nomsa"
os.chdir(test_dir)

In [ ]:
import subprocess, os, shutil

if os.path.exists("/content/srnas_ff"):
    shutil.rmtree("/content/srnas_ff")

result_clone = subprocess.run(
    ["git", "clone", "https://github.com/srnas/ff.git", "/content/srnas_ff"],
    capture_output=True,
    text=True
)
print("STDOUT:", result_clone.stdout)
print("STDERR:", result_clone.stderr)
print(f"Return code: {result_clone.returncode}")

if os.path.exists("/content/srnas_ff"):
    for item in os.listdir("/content/srnas_ff"):
        print(" -", item)

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os

test_path = "/content/drive/MyDrive/NEAT1_project"
if os.path.exists(test_path):
    for item in sorted(os.listdir(test_path))[:5]:
        print(" -", item)

In [ ]:
import os, subprocess, shutil

if not os.path.exists("/usr/local/miniconda"):
    !wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
    !bash miniconda.sh -b -p /usr/local/miniconda

os.environ["PATH"] = "/usr/local/miniconda/bin:" + os.environ["PATH"]

if not os.path.exists("/usr/local/miniconda/envs/gromacs"):
    !conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
    !conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
    !conda create -n gromacs -c bioconda -c conda-forge gromacs -y

gmx_bin = "/usr/local/miniconda/envs/gromacs/bin.AVX2_256/gmx"

dst_ff = "/usr/local/miniconda/envs/gromacs/share/gromacs/top/amber_na.ff"
if not os.path.exists(dst_ff):
    if os.path.exists("/content/srnas_ff"):
        shutil.rmtree("/content/srnas_ff")
    subprocess.run(["git", "clone", "https://github.com/srnas/ff.git", "/content/srnas_ff"], check=True)
    shutil.copytree("/content/srnas_ff/amber_na.ff", dst_ff)

test_dir = "/content/drive/MyDrive/NEAT1_project/MD_simulation/test_nomsa"
os.chdir(test_dir)

In [ ]:
import subprocess

result_grompp_prod = subprocess.run(
    [
        gmx_bin, "grompp",
        "-f", "production.mdp",
        "-c", "npt.part0002.gro",
        "-t", "npt.part0002.cpt",
        "-p", "topol.top",
        "-n", "index.ndx",
        "-o", "production.tpr",
        "-maxwarn", "2"
    ],
    capture_output=True,
    text=True
)

print("STDOUT:")
print(result_grompp_prod.stdout[-1500:])
print("\nSTDERR:")
print(result_grompp_prod.stderr[-2000:])
print(f"\nReturn code: {result_grompp_prod.returncode}")

In [ ]:
import os

for f in sorted(os.listdir(".")):
    if f.endswith(".cpt"):
        size_mb = os.path.getsize(f) / (1024*1024)
        mtime = os.path.getmtime(f)
        import datetime
        mtime_str = datetime.datetime.fromtimestamp(mtime).strftime("%Y-%m-%d %H:%M:%S")

In [ ]:
result_grompp_prod2 = subprocess.run(
    [
        gmx_bin, "grompp",
        "-f", "production.mdp",
        "-c", "npt.part0002.gro",
        "-t", "npt.cpt",
        "-p", "topol.top",
        "-n", "index.ndx",
        "-o", "production.tpr",
        "-maxwarn", "2"
    ],
    capture_output=True,
    text=True
)

print("STDOUT:")
print(result_grompp_prod2.stdout[-1000:])
print("\nSTDERR:")
print(result_grompp_prod2.stderr[-1500:])
print(f"\nReturn code: {result_grompp_prod2.returncode}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.chdir("/content")

import subprocess, shutil

if not os.path.exists("/usr/local/miniconda"):
    !wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
    !bash miniconda.sh -b -p /usr/local/miniconda

os.environ["PATH"] = "/usr/local/miniconda/bin:" + os.environ["PATH"]

if not os.path.exists("/usr/local/miniconda/envs/gromacs"):
    !conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
    !conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
    !conda create -n gromacs -c bioconda -c conda-forge gromacs -y

gmx_bin = "/usr/local/miniconda/envs/gromacs/bin.AVX2_256/gmx"

dst_ff = "/usr/local/miniconda/envs/gromacs/share/gromacs/top/amber_na.ff"
if not os.path.exists(dst_ff):
    if os.path.exists("/content/srnas_ff"):
        shutil.rmtree("/content/srnas_ff")
    subprocess.run(["git", "clone", "https://github.com/srnas/ff.git", "/content/srnas_ff"], check=True)
    shutil.copytree("/content/srnas_ff/amber_na.ff", dst_ff)

test_dir = "/content/drive/MyDrive/NEAT1_project/MD_simulation/test_nomsa"
os.chdir(test_dir)

for f in sorted(os.listdir(".")):
    if f.startswith("production"):
        size_mb = os.path.getsize(f) / (1024*1024)
        print(f" - {f} ({size_mb:.2f} MB)")

In [ ]:
with open("production.log") as f:
    log_content = f.read()

print(log_content)

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.chdir("/content")
import subprocess, shutil

if not os.path.exists("/usr/local/miniconda"):
    !wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
    !bash miniconda.sh -b -p /usr/local/miniconda
os.environ["PATH"] = "/usr/local/miniconda/bin:" + os.environ["PATH"]
if not os.path.exists("/usr/local/miniconda/envs/gromacs"):
    !conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
    !conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
    !conda create -n gromacs -c bioconda -c conda-forge gromacs -y
gmx_bin = "/usr/local/miniconda/envs/gromacs/bin.AVX2_256/gmx"
dst_ff = "/usr/local/miniconda/envs/gromacs/share/gromacs/top/amber_na.ff"
if not os.path.exists(dst_ff):
    if os.path.exists("/content/srnas_ff"):
        shutil.rmtree("/content/srnas_ff")
    subprocess.run(["git", "clone", "https://github.com/srnas/ff.git", "/content/srnas_ff"], check=True)
    shutil.copytree("/content/srnas_ff/amber_na.ff", dst_ff)

test_dir = "/content/drive/MyDrive/NEAT1_project/MD_simulation/test_nomsa"
os.chdir(test_dir)

import glob
cpt_files = glob.glob("production*.cpt")
for c in cpt_files:
    print(f"  {c}: {os.path.getmtime(c)}")

In [ ]:
import subprocess

cmd = (
    f"cd {test_dir} && nohup {gmx_bin} mdrun -deffnm production "
    f"-cpi production.cpt -noappend -nt 2 -cpt 5 "
    f"> production_nohup.log 2>&1 &"
)

subprocess.run(cmd, shell=True)

In [ ]:
import subprocess, os

result_trjconv_whole = subprocess.run(
    [gmx_bin, "trjconv", "-s", "production.part0003.tpr", "-f", "production.part0003.xtc",
     "-o", "production_whole.xtc", "-pbc", "mol", "-center"],
    input="1\n0\n",
    capture_output=True, text=True
)
print("STDOUT:", result_trjconv_whole.stdout[-800:])
print("STDERR:", result_trjconv_whole.stderr[-800:])
print(f"Return code: {result_trjconv_whole.returncode}")

In [ ]:
result_trjconv_whole = subprocess.run(
    [gmx_bin, "trjconv", "-s", "production.tpr", "-f", "production.part0003.xtc",
     "-o", "production_whole.xtc", "-pbc", "mol", "-center"],
    input="1\n0\n",
    capture_output=True, text=True
)
print("STDOUT:", result_trjconv_whole.stdout[-800:])
print("STDERR:", result_trjconv_whole.stderr[-1000:])
print(f"Return code: {result_trjconv_whole.returncode}")

In [ ]:
result_rmsd = subprocess.run(
    [gmx_bin, "rms", "-s", "production.tpr", "-f", "production_whole.xtc",
     "-o", "rmsd.xvg", "-tu", "ps"],
    input="1\n1\n",
    capture_output=True, text=True
)
print("STDOUT:", result_rmsd.stdout[-1000:])
print("STDERR:", result_rmsd.stderr[-800:])
print(f"Return code: {result_rmsd.returncode}")

In [ ]:
with open("rmsd.xvg") as f:
    lines = f.readlines()

data_lines = [l for l in lines if not l.startswith(("@", "#"))]

times = []
rmsds = []
for l in data_lines:
    parts = l.split()
    times.append(float(parts[0]))
    rmsds.append(float(parts[1]))

for t, r in zip(times, rmsds):
    print(f"  t={t:.1f} ps: RMSD={r:.4f} nm")

In [ ]:
import subprocess

result_rmsf = subprocess.run(
    [gmx_bin, "rmsf", "-s", "production.tpr", "-f", "production_whole.xtc",
     "-o", "rmsf.xvg", "-res"],
    input="1\n",
    capture_output=True, text=True
)
print("STDOUT:", result_rmsf.stdout[-800:])
print("STDERR:", result_rmsf.stderr[-500:])
print(f"Return code: {result_rmsf.returncode}")

In [ ]:
with open("rmsf.xvg") as f:
    lines = f.readlines()

data_lines = [l for l in lines if not l.startswith(("@", "#"))]

residues = []
rmsfs = []
for l in data_lines:
    parts = l.split()
    residues.append(int(float(parts[0])))
    rmsfs.append(float(parts[1]))

u_rich = set(range(14, 24))
a_rich = set(range(82, 102))

u_rich_rmsf = [r for res, r in zip(residues, rmsfs) if res in u_rich]
a_rich_rmsf = [r for res, r in zip(residues, rmsfs) if res in a_rich]
other_rmsf = [r for res, r in zip(residues, rmsfs) if res not in u_rich and res not in a_rich]

sorted_by_rmsf = sorted(zip(residues, rmsfs), key=lambda x: -x[1])
for res, r in sorted_by_rmsf[:10]:
    region = "U-rich" if res in u_rich else ("A-rich" if res in a_rich else "-")

In [ ]:
import subprocess

result_extract = subprocess.run(
    [gmx_bin, "trjconv", "-s", "production.tpr", "-f", "production_whole.xtc",
     "-o", "md_final_frame.pdb", "-dump", "100"],
    input="0\n",
    capture_output=True, text=True
)
print("STDOUT:", result_extract.stdout[-500:])
print("STDERR:", result_extract.stderr[-800:])
print(f"Return code: {result_extract.returncode}")

In [ ]:
result_extract_rna = subprocess.run(
    [gmx_bin, "trjconv", "-s", "production.tpr", "-f", "md_final_frame.pdb",
     "-o", "md_final_RNA_only.pdb"],
    input="1\n",
    capture_output=True, text=True
)
print("STDOUT:", result_extract_rna.stdout[-500:])
print("STDERR:", result_extract_rna.stderr[-500:])
print(f"Return code: {result_extract_rna.returncode}")

with open("md_final_RNA_only.pdb") as f:
    atom_count = sum(1 for line in f if line.startswith("ATOM"))

In [ ]:
!conda config --add channels defaults
!conda config --add channels bioconda
!conda config --add channels conda-forge

import shutil, os
if os.path.exists("/usr/local/miniconda/envs/fpocketR"):
    shutil.rmtree("/usr/local/miniconda/envs/fpocketR")

!conda create -n fpocketR fpocketr -y

In [ ]:
!/usr/local/miniconda/envs/fpocketR/bin/pip install "setuptools<81" --quiet

import subprocess, os

env_fix = os.environ.copy()
env_fix["MPLBACKEND"] = "Agg"

fpocketR_output_dir = "/content/drive/MyDrive/NEAT1_project/fpocketR_results"

result_fpocket_md = subprocess.run(
    [
        "/usr/local/miniconda/envs/fpocketR/bin/python", "-m", "fpocketR",
        "-pdb", "md_final_RNA_only.pdb",
        "-o", fpocketR_output_dir,
        "-n", "MD_final_100ps",
        "-y"
    ],
    env=env_fix,
    capture_output=True,
    text=True
)

print("STDOUT:")
print(result_fpocket_md.stdout[-2000:])
print(result_fpocket_md.stderr[-1000:])
print(f"\nReturn code: {result_fpocket_md.returncode}")

In [ ]:
with open("md_final_RNA_only.pdb") as f:
    lines = f.readlines()

for l in lines[:5]:
    print(repr(l))

In [ ]:
fixed_lines = []
for l in lines:
    if l.startswith("ATOM") or l.startswith("HETATM"):
        l = l[:21] + "A" + l[22:]
    fixed_lines.append(l)

with open("md_final_RNA_only_fixed.pdb", "w") as f:
    f.writelines(fixed_lines)

for l in fixed_lines:
    if l.startswith("ATOM"):
        print(repr(l))
        break

In [ ]:
import subprocess, os

env_fix = os.environ.copy()
env_fix["MPLBACKEND"] = "Agg"

result_fpocket_md2 = subprocess.run(
    [
        "/usr/local/miniconda/envs/fpocketR/bin/python", "-m", "fpocketR",
        "-pdb", "md_final_RNA_only_fixed.pdb",
        "-o", fpocketR_output_dir,
        "-n", "MD_final_100ps",
        "-y"
    ],
    env=env_fix,
    capture_output=True,
    text=True
)

print("STDOUT:")
print(result_fpocket_md2.stdout[-2000:])
print(result_fpocket_md2.stderr[-1200:])
print(f"\nReturn code: {result_fpocket_md2.returncode}")

In [ ]:
result_dir_md = os.path.join(fpocketR_output_dir, "MD_final_100ps_clean_out")

for root, dirs, files in os.walk(result_dir_md):
    for f in files:
        print(" -", os.path.relpath(os.path.join(root, f), result_dir_md))

info_path_md = os.path.join(result_dir_md, "MD_final_100ps_clean_info.txt")
with open(info_path_md) as f:
    print(f.read())

In [ ]:
pockets_dir_md = os.path.join(result_dir_md, "pockets")

for i in [1, 2, 3, 4, 5]:
    pocket_file = os.path.join(pockets_dir_md, f"pocket{i}_atm.pdb")
    residues = get_pocket_residues(pocket_file)

u_rich = set(range(14, 24))
a_rich = set(range(82, 102))

for i in [1, 2]:
    pocket_file = os.path.join(pockets_dir_md, f"pocket{i}_atm.pdb")
    residues = set(get_pocket_residues(pocket_file))

In [ ]:
def get_pocket_residues(pocket_pdb_path):
    residues = set()
    with open(pocket_pdb_path) as f:
        for line in f:
            if line.startswith("ATOM") or line.startswith("HETATM"):
                res_id = line[22:26].strip()
                residues.add(int(res_id))
    return sorted(residues)

for i in [1, 2, 3, 4, 5]:
    pocket_file = os.path.join(pockets_dir_md, f"pocket{i}_atm.pdb")
    residues = get_pocket_residues(pocket_file)

u_rich = set(range(14, 24))
a_rich = set(range(82, 102))

for i in [1, 2]:
    pocket_file = os.path.join(pockets_dir_md, f"pocket{i}_atm.pdb")
    residues = set(get_pocket_residues(pocket_file))

In [ ]:
import subprocess

make_ndx_commands = "ri 14-23\nri 82-101\nq\n"

result_ndx2 = subprocess.run(
    [gmx_bin, "make_ndx", "-f", "npt.part0002.gro", "-o", "index_regions.ndx"],
    input=make_ndx_commands,
    capture_output=True, text=True
)

print("STDOUT:")
print(result_ndx2.stdout[-2500:])
print(f"\nReturn code: {result_ndx2.returncode}")

In [ ]:
result_distance = subprocess.run(
    [gmx_bin, "distance", "-s", "production.tpr", "-f", "production_whole.xtc",
     "-n", "index_regions.ndx", "-select", "com of group r_14-23 plus com of group r_82-101",
     "-oall", "distance_Urich_Arich.xvg"],
    capture_output=True, text=True
)

print("STDOUT:")
print(result_distance.stdout[-1500:])
print("\nSTDERR:")
print(result_distance.stderr[-1500:])
print(f"\nReturn code: {result_distance.returncode}")

In [ ]:
result_distance2 = subprocess.run(
    [gmx_bin, "distance", "-s", "production.tpr", "-f", "production_whole.xtc",
     "-n", "index_regions.ndx", "-select", "com of group 11 plus com of group 12",
     "-oall", "distance_Urich_Arich.xvg"],
    capture_output=True, text=True
)

print("STDOUT:")
print(result_distance2.stdout[-1000:])
print("\nSTDERR:")
print(result_distance2.stderr[-1500:])
print(f"\nReturn code: {result_distance2.returncode}")

In [ ]:
with open("distance_Urich_Arich.xvg") as f:
    lines = f.readlines()

data_lines = [l for l in lines if not l.startswith(("@", "#"))]

times_d = []
distances = []
for l in data_lines:
    parts = l.split()
    times_d.append(float(parts[0]))
    distances.append(float(parts[1]))

for t, d in zip(times_d, distances):
    print(f"  {t:6.1f}  →  {d:.4f}")

n = len(distances)
first_half = distances[:n//2]
second_half = distances[n//2:]

In [ ]:
import subprocess

result_hbond = subprocess.run(
    [gmx_bin, "hbond", "-s", "production.tpr", "-f", "production_whole.xtc",
     "-n", "index_regions.ndx", "-num", "hbond_count.xvg"],
    input="11\n12\n",
    capture_output=True, text=True
)

print("STDOUT:")
print(result_hbond.stdout[-2000:])
print("\nSTDERR:")
print(result_hbond.stderr[-1500:])
print(f"\nReturn code: {result_hbond.returncode}")

In [ ]:
with open("hbond_count.xvg") as f:
    lines = f.readlines()

data_lines = [l for l in lines if not l.startswith(("@", "#"))]

times_hb = []
hbond_counts = []
for l in data_lines:
    parts = l.split()
    times_hb.append(float(parts[0]))
    hbond_counts.append(int(float(parts[1])))

for t, hb in zip(times_hb, hbond_counts):
    print(f"  {t:6.1f}  →  {hb}")

In [ ]:
production_extended_mdp = """; Production MD Extended (target 500 ps)
integrator      = md
nsteps          = 250000       ; 250000 x 2fs = 500 ps
dt              = 0.002

nstxout         = 5000
nstvout         = 5000
nstenergy       = 1000
nstlog          = 1000
nstxout-compressed = 1000

continuation    = yes
constraint_algorithm = lincs
constraints     = h-bonds

cutoff-scheme   = Verlet
nstlist         = 20
rcoulomb        = 1.0
rvdw            = 1.0
coulombtype     = PME

tcoupl          = V-rescale
tc-grps         = RNA Water_and_ions
tau_t           = 0.1     0.1
ref_t           = 300     300

pcoupl          = C-rescale
pcoupltype      = isotropic
tau_p           = 2.0
ref_p           = 1.0
compressibility = 4.5e-5

pbc             = xyz
gen_vel         = no
"""

with open("production_extended.mdp", "w") as f:
    f.write(production_extended_mdp)

import subprocess
result_grompp_ext = subprocess.run(
    [gmx_bin, "grompp", "-f", "production_extended.mdp",
     "-c", "production.part0003.gro", "-t", "production.cpt",
     "-p", "topol.top", "-n", "index.ndx",
     "-o", "production_ext.tpr", "-maxwarn", "2"],
    capture_output=True, text=True
)
print(result_grompp_ext.stderr[-800:])
print(f"Return code: {result_grompp_ext.returncode}")

In [ ]:
import subprocess

cmd_ext = (
    f"cd {test_dir} && nohup {gmx_bin} mdrun -deffnm production_ext "
    f"-nt 2 -cpt 5 "
    f"> production_ext_nohup.log 2>&1 &"
)

subprocess.run(cmd_ext, shell=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.chdir("/content")
import subprocess, shutil

if not os.path.exists("/usr/local/miniconda"):
    !wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
    !bash miniconda.sh -b -p /usr/local/miniconda
os.environ["PATH"] = "/usr/local/miniconda/bin:" + os.environ["PATH"]

if not os.path.exists("/usr/local/miniconda/envs/gromacs"):
    !conda config --add channels defaults
    !conda config --add channels bioconda
    !conda config --add channels conda-forge
    !conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
    !conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
    !conda create -n gromacs -c bioconda -c conda-forge gromacs -y

gmx_bin = "/usr/local/miniconda/envs/gromacs/bin.AVX2_256/gmx"
dst_ff = "/usr/local/miniconda/envs/gromacs/share/gromacs/top/amber_na.ff"
if not os.path.exists(dst_ff):
    if os.path.exists("/content/srnas_ff"):
        shutil.rmtree("/content/srnas_ff")
    subprocess.run(["git", "clone", "https://github.com/srnas/ff.git", "/content/srnas_ff"], check=True)
    shutil.copytree("/content/srnas_ff/amber_na.ff", dst_ff)

test_dir = "/content/drive/MyDrive/NEAT1_project/MD_simulation/test_nomsa"
os.chdir(test_dir)

import glob
for f in sorted(glob.glob("production_ext*")):
    print(" -", f, f"({os.path.getsize(f)/1024:.1f} KB)")

## 9. Docking (AutoDock Vina)

In [ ]:
import subprocess, os

vina_python = "/usr/local/miniconda/envs/vina/bin/python"
obabel_bin = "/usr/local/miniconda/envs/vina/bin/obabel"

result_receptor = subprocess.run(
    [obabel_bin, "md_final_RNA_only_fixed.pdb", "-O", "receptor.pdbqt",
     "-xr"],
    capture_output=True, text=True
)

print("STDOUT:", result_receptor.stdout)
print("STDERR:", result_receptor.stderr)
print(f"Return code: {result_receptor.returncode}")

if os.path.exists("receptor.pdbqt"):
    with open("receptor.pdbqt") as f:
        content = f.read()
    print("\n".join(content.split("\n")[:5]))

In [ ]:
ligands_smiles = {
    "quercetin": "O=c1c(O)c(-c2ccc(O)c(O)c2)oc2cc(O)cc(O)c12",
    "kanamycin_A": "NC1C(O)C(OC2C(N)CC(N)C(OC3OC(CO)C(O)C(O)C3N)C2O)OC(CN)C1O",
    "caffeine": "Cn1cnc2c1c(=O)n(C)c(=O)n2C"
}

os.makedirs("docking", exist_ok=True)
os.chdir("docking")

for name, smiles in ligands_smiles.items():
    with open(f"{name}.smi", "w") as f:
        f.write(smiles + "\n")

    result_lig = subprocess.run(
        [obabel_bin, f"{name}.smi", "-O", f"{name}.pdbqt", "--gen3d", "-p", "7.4"],
        capture_output=True, text=True
    )
    print(f"{name}: {result_lig.stderr.strip()}")

import os as _os
for f in sorted(_os.listdir(".")):
    print(" -", f)

In [ ]:
import numpy as np

pocket2_file = "/content/drive/MyDrive/NEAT1_project/fpocketR_results/MD_final_100ps_clean_out/pockets/pocket2_atm.pdb"

coords = []
with open(pocket2_file) as f:
    for line in f:
        if line.startswith("ATOM") or line.startswith("HETATM"):
            x = float(line[30:38])
            y = float(line[38:46])
            z = float(line[46:54])
            coords.append([x, y, z])

coords = np.array(coords)
center = coords.mean(axis=0)
size = coords.max(axis=0) - coords.min(axis=0) + 10

In [ ]:
vina_bin = "/usr/local/miniconda/envs/vina/bin/vina"

result_vina_quercetin = subprocess.run(
    [
        vina_bin,
        "--receptor", "../receptor.pdbqt",
        "--ligand", "quercetin.pdbqt",
        "--center_x", "86.62", "--center_y", "56.34", "--center_z", "40.66",
        "--size_x", "20", "--size_y", "24", "--size_z", "18",
        "--out", "quercetin_docked.pdbqt",
        "--log", "quercetin_docking.log",
        "--exhaustiveness", "16"
    ],
    capture_output=True, text=True
)

print("STDOUT:")
print(result_vina_quercetin.stdout)
print("\nSTDERR:")
print(result_vina_quercetin.stderr[-1000:])
print(f"\nReturn code: {result_vina_quercetin.returncode}")

In [ ]:
result_vina_quercetin2 = subprocess.run(
    [
        vina_bin,
        "--receptor", "../receptor.pdbqt",
        "--ligand", "quercetin.pdbqt",
        "--center_x", "86.62", "--center_y", "56.34", "--center_z", "40.66",
        "--size_x", "20", "--size_y", "24", "--size_z", "18",
        "--out", "quercetin_docked.pdbqt",
        "--exhaustiveness", "16"
    ],
    capture_output=True, text=True
)

print("STDOUT:")
print(result_vina_quercetin2.stdout)
print(result_vina_quercetin2.stderr[-1000:])
print(f"\nReturn code: {result_vina_quercetin2.returncode}")

In [ ]:
ligands_to_dock = ["kanamycin_A", "caffeine"]
docking_results = {"quercetin": -8.588}

for lig in ligands_to_dock:
    print(f"\n{'='*50}")
    print(f"Docking: {lig}")
    print('='*50)

    result = subprocess.run(
        [
            vina_bin,
            "--receptor", "../receptor.pdbqt",
            "--ligand", f"{lig}.pdbqt",
            "--center_x", "86.62", "--center_y", "56.34", "--center_z", "40.66",
            "--size_x", "20", "--size_y", "24", "--size_z", "18",
            "--out", f"{lig}_docked.pdbqt",
            "--exhaustiveness", "16"
        ],
        capture_output=True, text=True
    )

    print(result.stdout[-1200:])
    if result.returncode != 0:
        print("STDERR:", result.stderr[-800:])

    import re
    match = re.search(r"^\s*1\s+(-?\d+\.\d+)", result.stdout, re.MULTILINE)
    if match:
        docking_results[lig] = float(match.group(1))

for lig, score in docking_results.items():
    print(f"{lig:15s}: {score:.3f} kcal/mol")

In [ ]:
import subprocess, re

ligands_to_dock = ["kanamycin_A", "caffeine"]
docking_results = {"quercetin": -8.588}

for lig in ligands_to_dock:
    print(f"\n{'='*50}\nDocking: {lig}\n{'='*50}")

    result = subprocess.run(
        [
            vina_bin,
            "--receptor", "../receptor.pdbqt",
            "--ligand", f"{lig}.pdbqt",
            "--center_x", "86.62", "--center_y", "56.34", "--center_z", "40.66",
            "--size_x", "20", "--size_y", "24", "--size_z", "18",
            "--out", f"{lig}_docked.pdbqt",
            "--exhaustiveness", "16"
        ],
        capture_output=True, text=True
    )

    print(result.stdout[-1200:])
    if result.returncode != 0:
        print("STDERR:", result.stderr[-800:])

    match = re.search(r"^\s*1\s+(-?\d+\.\d+)", result.stdout, re.MULTILINE)
    if match:
        docking_results[lig] = float(match.group(1))

for lig, score in docking_results.items():
    print(f"{lig:15s}: {score:.3f} kcal/mol")

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, subprocess
os.environ["PATH"] = "/usr/local/miniconda/bin:" + os.environ["PATH"]

vina_bin = "/usr/local/miniconda/envs/vina/bin/vina"

docking_dir = "/content/drive/MyDrive/NEAT1_project/MD_simulation/test_nomsa/docking"
os.chdir(docking_dir)

script = f"""#!/bin/bash
cd {docking_dir}

{vina_bin} --receptor ../receptor.pdbqt --ligand kanamycin_A.pdbqt \\
  --center_x 86.62 --center_y 56.34 --center_z 40.66 \\
  --size_x 20 --size_y 24 --size_z 18 \\
  --out kanamycin_A_docked.pdbqt --exhaustiveness 16 > kanamycin_docking.log 2>&1

{vina_bin} --receptor ../receptor.pdbqt --ligand caffeine.pdbqt \\
  --center_x 86.62 --center_y 56.34 --center_z 40.66 \\
  --size_x 20 --size_y 24 --size_z 18 \\
  --out caffeine_docked.pdbqt --exhaustiveness 16 > caffeine_docking.log 2>&1

echo "DONE" > docking_status.txt
"""

with open("run_docking.sh", "w") as f:
    f.write(script)

subprocess.run(["chmod", "+x", "run_docking.sh"])
subprocess.run("nohup ./run_docking.sh > run_docking_nohup.log 2>&1 &", shell=True)

In [ ]:
import os

if os.path.exists("kanamycin_docking.log"):
    with open("kanamycin_docking.log") as f:
        print("\n--- Kanamycin log ---")
        print(f.read()[-800:])

if os.path.exists("caffeine_docking.log"):
    with open("caffeine_docking.log") as f:
        print("\n--- Caffeine log ---")
        print(f.read()[-800:])

In [ ]:
import os, subprocess

os.environ["PATH"] = "/usr/local/miniconda/bin:" + os.environ["PATH"]

if not os.path.exists("/usr/local/miniconda/envs/vina"):
    !conda config --add channels defaults
    !conda config --add channels bioconda
    !conda config --add channels conda-forge
    !conda create -n vina -c bioconda -c conda-forge vina openbabel -y

vina_bin = "/usr/local/miniconda/envs/vina/bin/vina"

In [ ]:
import os, subprocess

if not os.path.exists("/usr/local/miniconda"):
    !wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
    !bash miniconda.sh -b -p /usr/local/miniconda

os.environ["PATH"] = "/usr/local/miniconda/bin:" + os.environ["PATH"]

!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

!conda config --add channels defaults
!conda config --add channels bioconda
!conda config --add channels conda-forge

!conda create -n vina -c bioconda -c conda-forge vina openbabel -y

vina_bin = "/usr/local/miniconda/envs/vina/bin/vina"

In [ ]:
import os
docking_dir = "/content/drive/MyDrive/NEAT1_project/MD_simulation/test_nomsa/docking"
os.chdir(docking_dir)

import subprocess
subprocess.run("nohup ./run_docking.sh > run_docking_nohup.log 2>&1 &", shell=True)

In [ ]:
import os

for logf in ["kanamycin_docking.log", "caffeine_docking.log"]:
    if os.path.exists(logf):
        with open(logf) as f:
            content = f.read()
        print(f"\n--- {logf} ---")
        print(content[-1000:])

In [ ]:
import os
os.chdir("/content")

from google.colab import drive
drive.flush_and_unmount()

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
test_path = "/content/drive/MyDrive/NEAT1_project"

In [ ]:
import os, subprocess

if not os.path.exists("/usr/local/miniconda"):
    !wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
    !bash miniconda.sh -b -p /usr/local/miniconda

os.environ["PATH"] = "/usr/local/miniconda/bin:" + os.environ["PATH"]

!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!conda config --add channels defaults
!conda config --add channels bioconda
!conda config --add channels conda-forge

## 10. MM-GBSA

In [ ]:
import subprocess

cmd_install = (
    "cd /content && "
    "/usr/local/miniconda/bin/conda create -n gmxMMPBSA -c conda-forge -c bioconda "
    "gmx_MMPBSA ambertools -y "
    "> /content/drive/MyDrive/NEAT1_project/mmpbsa_install.log 2>&1"
)

subprocess.run(f"nohup bash -c '{cmd_install}' &", shell=True)

In [ ]:
import os

os.environ["PATH"] = "/usr/local/miniconda/bin:" + os.environ["PATH"]
mmpbsa_bin = "/usr/local/miniconda/envs/gmxMMPBSA/bin/gmx_MMPBSA"

gmx_bin = "/usr/local/miniconda/envs/gromacs/bin.AVX2_256/gmx"

if not os.path.exists(gmx_bin):
    !conda create -n gromacs -c bioconda -c conda-forge gromacs -y

    import subprocess, shutil
    dst_ff = "/usr/local/miniconda/envs/gromacs/share/gromacs/top/amber_na.ff"
    if not os.path.exists(dst_ff):
        if os.path.exists("/content/srnas_ff"):
            shutil.rmtree("/content/srnas_ff")
        subprocess.run(["git", "clone", "https://github.com/srnas/ff.git", "/content/srnas_ff"], check=True)
        shutil.copytree("/content/srnas_ff/amber_na.ff", dst_ff)

In [ ]:
import os
os.chdir(os.path.join(test_dir, "docking"))

acpype_bin = "/usr/local/miniconda/envs/gmxMMPBSA/bin/acpype"

if not os.path.exists(acpype_bin):
    import subprocess
    result = subprocess.run(["find", "/usr/local/miniconda/envs/gmxMMPBSA", "-name", "acpype*"],
                            capture_output=True, text=True)
    print(result.stdout)

In [ ]:
import subprocess
result = subprocess.run(
    ["/usr/local/miniconda/envs/gmxMMPBSA/bin/pip", "install", "acpype"],
    capture_output=True, text=True
)
print("STDOUT:", result.stdout[-1500:])
print("STDERR:", result.stderr[-800:])
print(f"Return code: {result.returncode}")

In [ ]:
import subprocess

obabel_bin = "/usr/local/miniconda/envs/vina/bin/obabel"

result_convert = subprocess.run(
    [obabel_bin, "quercetin_docked.pdbqt", "-O", "quercetin_pose1.mol2", "-f", "1", "-l", "1"],
    capture_output=True, text=True
)
print("STDOUT:", result_convert.stdout)
print("STDERR:", result_convert.stderr)

acpype_bin = "/usr/local/miniconda/envs/gmxMMPBSA/bin/acpype"
result_acpype = subprocess.run(
    [acpype_bin, "-i", "quercetin_pose1.mol2", "-c", "bcc", "-n", "0"],
    capture_output=True, text=True
)
print("\nACPYPE STDOUT:", result_acpype.stdout[-2000:])
print("ACPYPE STDERR:", result_acpype.stderr[-1000:])
print(f"Return code: {result_acpype.returncode}")

In [ ]:
result_convert2 = subprocess.run(
    [obabel_bin, "quercetin.smi", "-O", "quercetin_clean.mol2", "--gen3d", "-h"],
    capture_output=True, text=True
)
print("STDOUT:", result_convert2.stdout)
print("STDERR:", result_convert2.stderr)

result_formula = subprocess.run(
    [obabel_bin, "quercetin_clean.mol2", "-osmi", "--append", "formula"],
    capture_output=True, text=True
)

In [ ]:
result_acpype2 = subprocess.run(
    [acpype_bin, "-i", "quercetin_clean.mol2", "-c", "bcc", "-n", "0"],
    capture_output=True, text=True
)
print("STDOUT:", result_acpype2.stdout[-2000:])
print("STDERR:", result_acpype2.stderr[-1000:])
print(f"Return code: {result_acpype2.returncode}")

In [ ]:
import subprocess

result_hdf = subprocess.run(
    ["/usr/local/miniconda/bin/conda", "install", "-n", "gmxMMPBSA", "-c", "conda-forge", "hdf4", "-y"],
    capture_output=True, text=True
)
print(result_hdf.stdout[-1500:])
print(result_hdf.stderr[-500:])
print(f"Return code: {result_hdf.returncode}")

In [ ]:
import subprocess

result_find = subprocess.run(
    ["find", "/usr/local/miniconda/envs/gmxMMPBSA", "-name", "libmfhdf*"],
    capture_output=True, text=True
)
print(result_find.stdout)

In [ ]:
import os

os.environ["LD_LIBRARY_PATH"] = (
    "/usr/local/miniconda/envs/gmxMMPBSA/lib:" +
    "/usr/local/miniconda/envs/gmxMMPBSA/lib/python3.14/site-packages/acpype/amber_linux/lib:" +
    os.environ.get("LD_LIBRARY_PATH", "")
)

import subprocess
result_acpype3 = subprocess.run(
    [acpype_bin, "-i", "quercetin_clean.mol2", "-c", "bcc", "-n", "0"],
    capture_output=True, text=True,
    env=os.environ
)
print("STDOUT:", result_acpype3.stdout[-2000:])
print("STDERR:", result_acpype3.stderr[-1000:])
print(f"Return code: {result_acpype3.returncode}")

In [ ]:
import subprocess

cmd = (
    "export LD_LIBRARY_PATH=/usr/local/miniconda/envs/gmxMMPBSA/lib:"
    "/usr/local/miniconda/envs/gmxMMPBSA/lib/python3.14/site-packages/acpype/amber_linux/lib:$LD_LIBRARY_PATH && "
    f"cd {os.getcwd()} && "
    f"{acpype_bin} -i quercetin_clean.mol2 -c bcc -n 0"
)

result_acpype4 = subprocess.run(cmd, shell=True, capture_output=True, text=True, executable="/bin/bash")
print("STDOUT:", result_acpype4.stdout[-2000:])
print("STDERR:", result_acpype4.stderr[-1000:])
print(f"Return code: {result_acpype4.returncode}")

In [ ]:
import subprocess

subprocess.run(["cp", "/usr/local/miniconda/envs/gmxMMPBSA/lib/libmfhdf.so.0.0.0", "/usr/lib/x86_64-linux-gnu/"])
subprocess.run(["ln", "-sf", "/usr/lib/x86_64-linux-gnu/libmfhdf.so.0.0.0", "/usr/lib/x86_64-linux-gnu/libmfhdf.so.0"])

result_ldconfig = subprocess.run(["ldconfig"], capture_output=True, text=True)
print("ldconfig:", result_ldconfig.returncode, result_ldconfig.stderr)

result_test = subprocess.run(
    ["ldd", "/usr/local/miniconda/envs/gmxMMPBSA/lib/python3.14/site-packages/acpype/amber_linux/bin/teLeap"],
    capture_output=True, text=True
)
print("\nldd teLeap:")
print(result_test.stdout)

In [ ]:
import subprocess

result_find_tleap = subprocess.run(
    ["find", "/usr/local/miniconda/envs/gmxMMPBSA", "-name", "tleap", "-not", "-path", "*acpype*"],
    capture_output=True, text=True
)
print(result_find_tleap.stdout)

In [ ]:
import subprocess, os

env_fix = os.environ.copy()
env_fix["AMBERHOME"] = "/usr/local/miniconda/envs/gmxMMPBSA"
env_fix["PATH"] = "/usr/local/miniconda/envs/gmxMMPBSA/bin:" + env_fix.get("PATH", "")

import glob
for f in glob.glob("quercetin_clean_AC*") + glob.glob("quercetin_clean.acpype") + glob.glob("*.acpype", recursive=True):
    pass

result_acpype5 = subprocess.run(
    [acpype_bin, "-i", "quercetin_clean.mol2", "-c", "bcc", "-n", "0"],
    capture_output=True, text=True,
    env=env_fix
)
print("STDOUT:", result_acpype5.stdout[-2000:])
print("STDERR:", result_acpype5.stderr[-1000:])
print(f"Return code: {result_acpype5.returncode}")

In [ ]:
import os

acpype_out_dir = "quercetin_clean.acpype"
if os.path.exists(acpype_out_dir):
    for f in sorted(os.listdir(acpype_out_dir)):
        size_kb = os.path.getsize(os.path.join(acpype_out_dir, f)) / 1024
        print(f" - {f} ({size_kb:.1f} KB)")
else:
    import glob
    for d in glob.glob("*acpype*"):
        print(" -", d)

In [ ]:
import subprocess

obabel_bin_gmx = "/usr/local/miniconda/envs/vina/bin/obabel"
result_pose_pdb = subprocess.run(
    [obabel_bin_gmx, "quercetin_docked.pdbqt", "-O", "quercetin_pose1.pdb", "-f", "1", "-l", "1"],
    capture_output=True, text=True
)

with open("quercetin_pose1.pdb") as f:
    pose_atoms = [l.split()[2] for l in f if l.startswith(("ATOM","HETATM"))]

with open("quercetin_clean.acpype/quercetin_clean_NEW.pdb") as f:
    acpype_atoms = [l.split()[2] for l in f if l.startswith(("ATOM","HETATM"))]

In [ ]:
import subprocess

result_addH = subprocess.run(
    [obabel_bin_gmx, "quercetin_pose1.pdb", "-O", "quercetin_pose1_H.pdb", "-h"],
    capture_output=True, text=True
)
print("STDOUT:", result_addH.stdout)
print("STDERR:", result_addH.stderr)

with open("quercetin_pose1_H.pdb") as f:
    pose_h_atoms = [l.split()[2] for l in f if l.startswith(("ATOM","HETATM"))]
print(pose_h_atoms)

In [ ]:
def get_elements_in_order(filepath):
    elements = []
    with open(filepath) as f:
        for l in f:
            if l.startswith(("ATOM", "HETATM")):
                elem = l[76:78].strip() if len(l) > 77 else l.split()[-1]
                elements.append(elem)
    return elements

pose_elements = get_elements_in_order("quercetin_pose1_H.pdb")
acpype_elements = get_elements_in_order("quercetin_clean.acpype/quercetin_clean_NEW.pdb")

print("\nacpype:            ", acpype_elements)

In [ ]:
import subprocess, os

result_pose_mol2 = subprocess.run(
    [obabel_bin_gmx, "quercetin_pose1_H.pdb", "-O", "quercetin_pose1_final.mol2"],
    capture_output=True, text=True
)

result_formula2 = subprocess.run(
    [obabel_bin_gmx, "quercetin_pose1_final.mol2", "-osmi", "--append", "formula"],
    capture_output=True, text=True
)

env_fix = os.environ.copy()
env_fix["AMBERHOME"] = "/usr/local/miniconda/envs/gmxMMPBSA"
env_fix["PATH"] = "/usr/local/miniconda/envs/gmxMMPBSA/bin:" + env_fix.get("PATH", "")

result_acpype_final = subprocess.run(
    [acpype_bin, "-i", "quercetin_pose1_final.mol2", "-c", "bcc", "-n", "0"],
    capture_output=True, text=True, env=env_fix
)
print("\nACPYPE:", result_acpype_final.stdout[-1000:])
print("Return code:", result_acpype_final.returncode)

In [ ]:
import os

acpype_final_dir = "quercetin_pose1_final.acpype"
for f in sorted(os.listdir(acpype_final_dir)):
    print(" -", f)

In [ ]:
import os

rna_path = os.path.join(test_dir, "md_final_RNA_only_fixed.pdb")
with open(rna_path) as f:
    rna_lines = [l for l in f if l.startswith("ATOM")]

ligand_path = os.path.join(acpype_final_dir, "quercetin_pose1_final_NEW.pdb")
with open(ligand_path) as f:
    ligand_lines_raw = [l for l in f if l.startswith(("ATOM", "HETATM"))]

ligand_lines = []
for i, l in enumerate(ligand_lines_raw):
    new_line = "HETATM" + l[6:17] + "LIG" + " B" + "102" + l[26:]
    ligand_lines.append(new_line)

complex_path = "complex_RNA_quercetin.pdb"
with open(complex_path, "w") as f:
    f.writelines(rna_lines)
    f.write("TER\n")
    f.writelines(ligand_lines)
    f.write("END\n")

with open(complex_path) as f:
    all_lines = f.readlines()
for l in all_lines[len(rna_lines)+1:len(rna_lines)+4]:
    print(repr(l))

In [ ]:
def format_pdb_line(atom_serial, atom_name, res_name, chain, res_num, x, y, z, element):
    atom_name_fmt = f" {atom_name:<3}" if len(atom_name) < 4 else atom_name
    return (f"HETATM{atom_serial:>5} {atom_name_fmt:<4}{res_name:>3} {chain}{res_num:>4}    "
            f"{x:8.3f}{y:8.3f}{z:8.3f}  1.00  0.00          {element:>2}\n")

ligand_fixed_lines = []
for i, l in enumerate(ligand_lines_raw):
    atom_name = l[12:16].strip()
    x, y, z = float(l[30:38]), float(l[38:46]), float(l[46:54])
    element = l[76:78].strip() if len(l) > 77 else atom_name[0]
    serial = 3227 + i + 1
    ligand_fixed_lines.append(format_pdb_line(serial, atom_name, "LIG", "B", 102, x, y, z, element))

complex_path = "complex_RNA_quercetin.pdb"
with open(complex_path, "w") as f:
    f.writelines(rna_lines)
    f.write("TER\n")
    f.writelines(ligand_fixed_lines)
    f.write("END\n")

for l in ligand_fixed_lines[:3]:
    print(repr(l))

In [ ]:
import shutil

ligand_itp_src = os.path.join(acpype_final_dir, "quercetin_pose1_final_GMX.itp")
shutil.copy(ligand_itp_src, os.path.join(test_dir, "LIG.itp"))
shutil.copy(os.path.join(acpype_final_dir, "posre_quercetin_pose1_final.itp"),
            os.path.join(test_dir, "posre_LIG.itp"))

with open(os.path.join(test_dir, "LIG.itp")) as f:
    lig_itp_content = f.read()

import re
match = re.search(r"\[ moleculetype \].*?\n(.*?)\n", lig_itp_content, re.DOTALL)

In [ ]:
sections = re.findall(r"\[ (\w+) \]", lig_itp_content)

mol_idx = lig_itp_content.find("[ moleculetype ]")
print(lig_itp_content[mol_idx:mol_idx+200])

In [ ]:
os.chdir(test_dir)

atomtypes_section = lig_itp_content[:lig_itp_content.find("[ moleculetype ]")]
rest_of_itp = lig_itp_content[lig_itp_content.find("[ moleculetype ]"):]

with open("LIG_atomtypes.itp", "w") as f:
    f.write(atomtypes_section)
with open("LIG.itp", "w") as f:
    f.write(rest_of_itp)

with open("topol.top") as f:
    top_content = f.read()

top_sections_idx = [(m.group(1), m.start()) for m in re.finditer(r"\[ (\w+) \]", top_content)]
for name, idx in top_sections_idx:
    print(f"  {name} @ {idx}")

include_end = top_content.rfind("#include")
include_end = top_content.find("\n", include_end) + 1
molecules_idx = top_content.find("[ molecules ]")

In [ ]:
ff_include_end = top_content.find("\n", top_content.find("#include")) + 1

new_top = (
    top_content[:ff_include_end] +
    '#include "LIG_atomtypes.itp"\n' +
    top_content[ff_include_end:include_end] +
    '#include "LIG.itp"\n' +
    top_content[include_end:]
)

new_top = new_top.rstrip() + "\nquercetin_pose1_final       1\n"

with open("topol_complex.top", "w") as f:
    f.write(new_top)

print("\n".join(new_top.split("\n")[:5]))
print("\n".join(new_top.split("\n")[-8:]))

In [ ]:
new_top_novacuum = new_top.split("[ molecules ]")[0] + "[ molecules ]\n; Compound        #mols\nRNA_chain_0         1\nquercetin_pose1_final       1\n"

with open("topol_complex_novacuum.top", "w") as f:
    f.write(new_top_novacuum)

result_pdb2gro = subprocess.run(
    [gmx_bin, "editconf", "-f", "complex_RNA_quercetin.pdb", "-o", "complex_RNA_quercetin.gro"],
    capture_output=True, text=True
)
print("\nSTDOUT:", result_pdb2gro.stdout[-500:])
print("STDERR:", result_pdb2gro.stderr[-500:])

In [ ]:
import shutil

shutil.copy("docking/complex_RNA_quercetin.pdb", "complex_RNA_quercetin.pdb")

result_pdb2gro = subprocess.run(
    [gmx_bin, "editconf", "-f", "complex_RNA_quercetin.pdb", "-o", "complex_RNA_quercetin.gro"],
    capture_output=True, text=True
)
print("\nSTDOUT:", result_pdb2gro.stdout[-500:])
print("STDERR:", result_pdb2gro.stderr[-800:])
print(f"Return code: {result_pdb2gro.returncode}")

In [ ]:
result_grompp_test = subprocess.run(
    [gmx_bin, "grompp", "-f", "em.mdp",
     "-c", "complex_RNA_quercetin.gro",
     "-p", "topol_complex_novacuum.top",
     "-o", "complex_test.tpr",
     "-maxwarn", "5"],
    capture_output=True, text=True
)
print("STDOUT:", result_grompp_test.stdout[-1500:])
print("STDERR:", result_grompp_test.stderr[-2500:])
print(f"Return code: {result_grompp_test.returncode}")

In [ ]:
result_box_complex = subprocess.run(
    [gmx_bin, "editconf", "-f", "complex_RNA_quercetin.pdb",
     "-o", "complex_boxed.gro", "-c", "-d", "1.5", "-bt", "cubic"],
    capture_output=True, text=True
)
print(result_box_complex.stdout[-800:])
print(f"Return code: {result_box_complex.returncode}")

result_grompp_test2 = subprocess.run(
    [gmx_bin, "grompp", "-f", "em.mdp",
     "-c", "complex_boxed.gro",
     "-p", "topol_complex_novacuum.top",
     "-o", "complex_test.tpr",
     "-maxwarn", "5"],
    capture_output=True, text=True
)
print(result_grompp_test2.stderr[-1200:])
print(f"Return code: {result_grompp_test2.returncode}")

In [ ]:
import time

start_time = time.time()

result_em_complex = subprocess.run(
    [gmx_bin, "mdrun", "-deffnm", "complex_test", "-nt", "2"],
    capture_output=True, text=True
)

elapsed = time.time() - start_time
print(result_em_complex.stderr[-1500:])
print(f"Return code: {result_em_complex.returncode}")

In [ ]:
result_traj = subprocess.run(
    [gmx_bin, "trjconv", "-f", "complex_test.gro", "-s", "complex_test.tpr",
     "-o", "complex_singleframe.xtc"],
    input="0\n",
    capture_output=True, text=True
)
print("trjconv:", result_traj.returncode)

result_ndx_complex = subprocess.run(
    [gmx_bin, "make_ndx", "-f", "complex_test.gro", "-o", "index_complex.ndx"],
    input="q\n",
    capture_output=True, text=True
)
print(result_ndx_complex.stdout[-1500:])

In [ ]:
mmpbsa_input = """&general
sys_name="RNA-Quercetin"
startframe=1
endframe=1
interval=1
forcefields="oldff/leaprc.ff99SB,leaprc.gaff2"
/
&gb
igb=5
saltcon=0.150
/
"""

with open("mmpbsa.in", "w") as f:
    f.write(mmpbsa_input)

mmpbsa_bin = "/usr/local/miniconda/envs/gmxMMPBSA/bin/gmx_MMPBSA"

env_fix = os.environ.copy()
env_fix["AMBERHOME"] = "/usr/local/miniconda/envs/gmxMMPBSA"
env_fix["PATH"] = "/usr/local/miniconda/envs/gmxMMPBSA/bin:" + env_fix.get("PATH", "")

result_mmpbsa = subprocess.run(
    [mmpbsa_bin, "-O", "-i", "mmpbsa.in",
     "-cs", "complex_test.tpr", "-ci", "index_complex.ndx", "-cg", "1", "2",
     "-ct", "complex_singleframe.xtc", "-cp", "topol_complex_novacuum.top",
     "-o", "FINAL_RESULTS_MMPBSA.dat", "-eo", "FINAL_RESULTS_MMPBSA.csv"],
    capture_output=True, text=True, env=env_fix
)
print("STDOUT:", result_mmpbsa.stdout[-2500:])
print("STDERR:", result_mmpbsa.stderr[-1500:])
print(f"Return code: {result_mmpbsa.returncode}")

In [ ]:
import os

env_fix = os.environ.copy()
env_fix["AMBERHOME"] = "/usr/local/miniconda/envs/gmxMMPBSA"
env_fix["PATH"] = (
    "/usr/local/miniconda/envs/gmxMMPBSA/bin:" +
    "/usr/local/miniconda/envs/gromacs/bin.AVX2_256:" +
    env_fix.get("PATH", "")
)

import subprocess
test_which = subprocess.run(["which", "gmx"], capture_output=True, text=True, env=env_fix)

In [ ]:
result_mmpbsa2 = subprocess.run(
    [mmpbsa_bin, "-O", "-i", "mmpbsa.in",
     "-cs", "complex_test.tpr", "-ci", "index_complex.ndx", "-cg", "1", "2",
     "-ct", "complex_singleframe.xtc", "-cp", "topol_complex_novacuum.top",
     "-o", "FINAL_RESULTS_MMPBSA.dat", "-eo", "FINAL_RESULTS_MMPBSA.csv"],
    capture_output=True, text=True, env=env_fix
)
print("STDOUT:", result_mmpbsa2.stdout[-2500:])
print("STDERR:", result_mmpbsa2.stderr[-2000:])
print(f"Return code: {result_mmpbsa2.returncode}")

In [ ]:
import os

if os.path.exists("FINAL_RESULTS_MMPBSA.dat"):
    with open("FINAL_RESULTS_MMPBSA.dat") as f:
        content = f.read()
    print(content)

In [ ]:
import datetime
cpt_time = os.path.getmtime("production_ext.cpt")

In [ ]:
import os

os.environ["PATH"] = "/usr/local/miniconda/bin:" + os.environ["PATH"]

import subprocess
test_conda = subprocess.run(["/usr/local/miniconda/bin/conda", "--version"], capture_output=True, text=True)
print("conda version test:", test_conda.stdout, test_conda.stderr)

In [ ]:
import shutil, os

if os.path.exists("/usr/local/miniconda"):
    shutil.rmtree("/usr/local/miniconda")

!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /content/miniconda.sh
!bash /content/miniconda.sh -b -p /usr/local/miniconda

import subprocess
test_conda = subprocess.run(["/usr/local/miniconda/bin/conda", "--version"], capture_output=True, text=True)

In [ ]:
import shutil, os

src = "/mnt/user-data/uploads/NEAT1_ENE_A.pdb"
dst = "/content/drive/MyDrive/NEAT1_project/NEAT1_ENE_A_RNAComposer.pdb"
shutil.copy(src, dst)

with open(dst) as f:
    lines = f.readlines()
res_ids = set()
for l in lines:
    if l.startswith("ATOM"):
        res_ids.add(int(l[22:26].strip()))

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import shutil, os

src = "/content/NEAT1_ENE_A.pdb"
dst = "/content/drive/MyDrive/NEAT1_project/NEAT1_ENE_A_RNAComposer.pdb"

shutil.copy(src, dst)

with open(dst) as f:
    lines = f.readlines()
res_ids = set()
for l in lines:
    if l.startswith("ATOM"):
        res_ids.add(int(l[22:26].strip()))

In [ ]:
import sys

!{sys.executable} -m pip install biopython --quiet

from Bio.PDB import PDBParser, Superimposer

In [ ]:
from Bio.PDB import PDBParser, Superimposer

parser = PDBParser(QUIET=True)

def get_c1_coords(pdb_path):
    struct = parser.get_structure("s", pdb_path)
    atoms = []
    for model in struct:
        for chain in model:
            for residue in chain:
                if "C1'" in residue:
                    atoms.append(residue["C1'"])
        break
    return atoms

paths = {
    "RhoFold_noMSA": "/content/drive/MyDrive/NEAT1_project/NEAT1_ENE_A_RhoFold.pdb",
    "RhoFold_withMSA": "/content/drive/MyDrive/NEAT1_project/NEAT1_ENE_A_RhoFold_withMSA.pdb",
    "trRosettaRNA": "/content/drive/MyDrive/NEAT1_project/trRosettaRNA_results/RNA031693_results/model1.pdb",
    "RNAComposer": "/content/drive/MyDrive/NEAT1_project/NEAT1_ENE_A_RNAComposer.pdb"
}

import itertools
print("-"*55)
for (name1, path1), (name2, path2) in itertools.combinations(paths.items(), 2):
    a1 = get_c1_coords(path1)
    a2 = get_c1_coords(path2)
    sup = Superimposer()
    sup.set_atoms(a1, a2)
    print(f"{name1:<20}{name2:<20}{sup.rms:.2f}\n")

In [ ]:
import os
os.environ["PATH"] = "/usr/local/miniconda/bin:" + os.environ["PATH"]

fpocketR_bin_check = "/usr/local/miniconda/envs/fpocketR/bin/python"

if not os.path.exists(fpocketR_bin_check):
    !conda config --add channels defaults
    !conda config --add channels bioconda
    !conda config --add channels conda-forge
    !conda create -n fpocketR fpocketr -y
    !/usr/local/miniconda/envs/fpocketR/bin/pip install "setuptools<81" --quiet

In [ ]:
import os

rnacomposer_path = "/content/drive/MyDrive/NEAT1_project/NEAT1_ENE_A_RNAComposer.pdb"
with open(rnacomposer_path) as f:
    lines = f.readlines()

for l in lines:
    if l.startswith("ATOM"):
        print(repr(l))
        break

In [ ]:
import subprocess, os

env_fix = os.environ.copy()
env_fix["MPLBACKEND"] = "Agg"

fpocketR_output_dir = "/content/drive/MyDrive/NEAT1_project/fpocketR_results"

result_fpocket_rnacomposer = subprocess.run(
    [
        "/usr/local/miniconda/envs/fpocketR/bin/python", "-m", "fpocketR",
        "-pdb", rnacomposer_path,
        "-o", fpocketR_output_dir,
        "-n", "RNAComposer",
        "-y"
    ],
    env=env_fix,
    capture_output=True,
    text=True
)

print("STDOUT:")
print(result_fpocket_rnacomposer.stdout[-2000:])
print(result_fpocket_rnacomposer.stderr[-1000:])
print(f"\nReturn code: {result_fpocket_rnacomposer.returncode}")

In [ ]:
result_dir_rc = os.path.join(fpocketR_output_dir, "RNAComposer_clean_out")

info_path_rc = os.path.join(result_dir_rc, "RNAComposer_clean_info.txt")
if os.path.exists(info_path_rc):
    with open(info_path_rc) as f:
        print(f.read())
else:
    for f in os.listdir(result_dir_rc):
        print(" -", f)

In [ ]:
def get_pocket_residues(pocket_pdb_path):
    residues = set()
    with open(pocket_pdb_path) as f:
        for line in f:
            if line.startswith("ATOM") or line.startswith("HETATM"):
                res_id = line[22:26].strip()
                residues.add(int(res_id))
    return sorted(residues)

pockets_dir_rc = os.path.join(result_dir_rc, "pockets")
pocket1_rc_residues = get_pocket_residues(os.path.join(pockets_dir_rc, "pocket1_atm.pdb"))

u_rich = set(range(14, 24))
a_rich = set(range(82, 102))
pocket_set = set(pocket1_rc_residues)

In [ ]:
import os

if not os.path.exists('/usr/local/miniconda/bin/conda'):
    import shutil
    if os.path.exists("/usr/local/miniconda"):
        shutil.rmtree("/usr/local/miniconda")
    !wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /content/miniconda.sh
    !bash /content/miniconda.sh -b -p /usr/local/miniconda

    import subprocess
    test = subprocess.run(["/usr/local/miniconda/bin/conda", "--version"], capture_output=True, text=True)

In [ ]:
import os
os.environ["PATH"] = "/usr/local/miniconda/bin:" + os.environ["PATH"]

!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!conda config --add channels defaults
!conda config --add channels bioconda
!conda config --add channels conda-forge
!conda create -n gromacs -c bioconda -c conda-forge gromacs -y

In [ ]:
import glob, os

cpt_files = sorted(glob.glob("production_ext*.cpt"), key=os.path.getmtime)
for c in cpt_files:
    print(" -", c, f"({os.path.getmtime(c)})")

In [ ]:
import os, subprocess, shutil

os.environ["PATH"] = "/usr/local/miniconda/bin:" + os.environ["PATH"]

if not os.path.exists("/usr/local/miniconda/bin/conda"):
    if os.path.exists("/usr/local/miniconda"):
        shutil.rmtree("/usr/local/miniconda")
    !wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /content/miniconda.sh
    !bash /content/miniconda.sh -b -p /usr/local/miniconda

gmx_bin = "/usr/local/miniconda/envs/gromacs/bin.AVX2_256/gmx"
if not os.path.exists(gmx_bin):
    !conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
    !conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
    !conda config --add channels defaults
    !conda config --add channels bioconda
    !conda config --add channels conda-forge
    !conda create -n gromacs -c bioconda -c conda-forge gromacs -y

dst_ff = "/usr/local/miniconda/envs/gromacs/share/gromacs/top/amber_na.ff"
if not os.path.exists(dst_ff):
    if os.path.exists("/content/srnas_ff"):
        shutil.rmtree("/content/srnas_ff")
    subprocess.run(["git", "clone", "https://github.com/srnas/ff.git", "/content/srnas_ff"], check=True)
    shutil.copytree("/content/srnas_ff/amber_na.ff", dst_ff)

test_dir = "/content/drive/MyDrive/NEAT1_project/MD_simulation/test_nomsa"
os.chdir(test_dir)

import glob
xtc_parts = sorted(glob.glob("production_ext.part*.xtc"))

result_trjcat = subprocess.run(
    [gmx_bin, "trjcat", "-f"] + xtc_parts + ["-o", "production_ext_full.xtc", "-cat"],
    capture_output=True, text=True
)
print(result_trjcat.stderr[-1000:])
print(f"Return code: {result_trjcat.returncode}")

In [ ]:
import subprocess

result_whole_full = subprocess.run(
    [gmx_bin, "trjconv", "-s", "production_ext.tpr", "-f", "production_ext_full.xtc",
     "-o", "production_ext_full_whole.xtc", "-pbc", "mol", "-center"],
    input="1\n0\n",
    capture_output=True, text=True
)
print("trjconv:", result_whole_full.returncode)

result_rmsd_full = subprocess.run(
    [gmx_bin, "rms", "-s", "production_ext.tpr", "-f", "production_ext_full_whole.xtc",
     "-o", "rmsd_full.xvg", "-tu", "ps"],
    input="1\n1\n",
    capture_output=True, text=True
)
print("rms:", result_rmsd_full.returncode)
print(result_rmsd_full.stderr[-500:])

In [ ]:
with open("rmsd_full.xvg") as f:
    lines = f.readlines()

data_lines = [l for l in lines if not l.startswith(("@", "#"))]

times_full = []
rmsds_full = []
for l in data_lines:
    parts = l.split()
    times_full.append(float(parts[0]))
    rmsds_full.append(float(parts[1]))

for i in range(0, len(times_full), 10):
    print(f"  t={times_full[i]:.1f} ps: RMSD={rmsds_full[i]:.4f} nm")

n = len(rmsds_full)
q1_slope = (rmsds_full[n//4] - rmsds_full[0]) / (times_full[n//4] - times_full[0])
q4_slope = (rmsds_full[-1] - rmsds_full[3*n//4]) / (times_full[-1] - times_full[3*n//4])

In [ ]:
import os

if not os.path.exists("index_regions.ndx"):
    make_ndx_commands = "ri 14-23\nri 82-101\nq\n"
    result_ndx = subprocess.run(
        [gmx_bin, "make_ndx", "-f", "npt.part0002.gro", "-o", "index_regions.ndx"],
        input=make_ndx_commands,
        capture_output=True, text=True
    )
    print(result_ndx.stdout[-800:])

In [ ]:
result_distance_full = subprocess.run(
    [gmx_bin, "distance", "-s", "production_ext.tpr", "-f", "production_ext_full_whole.xtc",
     "-n", "index_regions.ndx", "-select", "com of group 11 plus com of group 12",
     "-oall", "distance_full.xvg"],
    capture_output=True, text=True
)
print(result_distance_full.stdout[-1200:])
print(f"Return code: {result_distance_full.returncode}")

In [ ]:
with open("distance_full.xvg") as f:
    lines = f.readlines()

data_lines = [l for l in lines if not l.startswith(("@", "#"))]

times_d_full = []
distances_full = []
for l in data_lines:
    parts = l.split()
    times_d_full.append(float(parts[0]))
    distances_full.append(float(parts[1]))

for i in range(0, len(times_d_full), 10):
    print(f"  t={times_d_full[i]:.1f} ps: distance={distances_full[i]:.4f} nm")
print(f"  t={times_d_full[-1]:.1f} ps: distance={distances_full[-1]:.4f} nm")

n = len(distances_full)
q1_mean = sum(distances_full[:n//4]) / (n//4)
q4_mean = sum(distances_full[3*n//4:]) / (n - 3*n//4)

In [ ]:
import subprocess, shutil, time

gmx_bin = "/usr/local/miniconda/envs/gromacs/bin.AVX2_256/gmx"

if not os.path.exists(gmx_bin):
    if not os.path.exists("/usr/local/miniconda/bin/conda"):
        if os.path.exists("/usr/local/miniconda"):
            shutil.rmtree("/usr/local/miniconda")
        !wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /content/miniconda.sh
        !bash /content/miniconda.sh -b -p /usr/local/miniconda
    !conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
    !conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
    !conda config --add channels defaults
    !conda config --add channels bioconda
    !conda config --add channels conda-forge
    !conda create -n gromacs -c bioconda -c conda-forge gromacs -y

dst_ff = "/usr/local/miniconda/envs/gromacs/share/gromacs/top/amber_na.ff"
if not os.path.exists(dst_ff):
    if os.path.exists("/content/srnas_ff"):
        shutil.rmtree("/content/srnas_ff")
    subprocess.run(["git", "clone", "https://github.com/srnas/ff.git", "/content/srnas_ff"], check=True)
    shutil.copytree("/content/srnas_ff/amber_na.ff", dst_ff)

cmd_continue = (
    f"cd {test_dir} && nohup {gmx_bin} mdrun -deffnm production_ext "
    f"-cpi production_ext.cpt -noappend -nt 2 -cpt 5 "
    f"> production_ext_nohup_latest.log 2>&1 &"
)
subprocess.run(cmd_continue, shell=True)
time.sleep(5)
check = subprocess.run(["pgrep", "-af", "mdrun"], capture_output=True, text=True)

In [ ]:
import os, subprocess, shutil

os.environ["PATH"] = "/usr/local/miniconda/bin:" + os.environ["PATH"]
gmx_bin = "/usr/local/miniconda/envs/gromacs/bin.AVX2_256/gmx"

if not os.path.exists(gmx_bin):
    if not os.path.exists("/usr/local/miniconda/bin/conda"):
        if os.path.exists("/usr/local/miniconda"):
            shutil.rmtree("/usr/local/miniconda")
        !wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /content/miniconda.sh
        !bash /content/miniconda.sh -b -p /usr/local/miniconda
    !conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
    !conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
    !conda config --add channels defaults
    !conda config --add channels bioconda
    !conda config --add channels conda-forge
    !conda create -n gromacs -c bioconda -c conda-forge gromacs -y

dst_ff = "/usr/local/miniconda/envs/gromacs/share/gromacs/top/amber_na.ff"
if not os.path.exists(dst_ff):
    if os.path.exists("/content/srnas_ff"):
        shutil.rmtree("/content/srnas_ff")
    subprocess.run(["git", "clone", "https://github.com/srnas/ff.git", "/content/srnas_ff"], check=True)
    shutil.copytree("/content/srnas_ff/amber_na.ff", dst_ff)

test_dir = "/content/drive/MyDrive/NEAT1_project/MD_simulation/test_nomsa"
os.chdir(test_dir)

import glob
xtc_parts_final = sorted(glob.glob("production_ext.part*.xtc"))

result_trjcat_final = subprocess.run(
    [gmx_bin, "trjcat", "-f"] + xtc_parts_final + ["-o", "production_ext_500ps_full.xtc", "-cat"],
    capture_output=True, text=True
)
print(result_trjcat_final.stderr[-800:])
print(f"Return code: {result_trjcat_final.returncode}")

In [ ]:
import subprocess

result_whole_500 = subprocess.run(
    [gmx_bin, "trjconv", "-s", "production_ext.tpr", "-f", "production_ext_500ps_full.xtc",
     "-o", "production_ext_500ps_whole.xtc", "-pbc", "mol", "-center"],
    input="1\n0\n",
    capture_output=True, text=True
)
print("trjconv:", result_whole_500.returncode)

result_rmsd_500 = subprocess.run(
    [gmx_bin, "rms", "-s", "production_ext.tpr", "-f", "production_ext_500ps_whole.xtc",
     "-o", "rmsd_500ps.xvg", "-tu", "ps"],
    input="1\n1\n",
    capture_output=True, text=True
)
print("rms:", result_rmsd_500.returncode)

import os
if not os.path.exists("index_regions.ndx"):
    make_ndx_commands = "ri 14-23\nri 82-101\nq\n"
    subprocess.run([gmx_bin, "make_ndx", "-f", "npt.part0002.gro", "-o", "index_regions.ndx"],
                   input=make_ndx_commands, capture_output=True, text=True)

result_distance_500 = subprocess.run(
    [gmx_bin, "distance", "-s", "production_ext.tpr", "-f", "production_ext_500ps_whole.xtc",
     "-n", "index_regions.ndx", "-select", "com of group 11 plus com of group 12",
     "-oall", "distance_500ps.xvg"],
    capture_output=True, text=True
)
print("distance:", result_distance_500.stdout[-800:])

In [ ]:
with open("rmsd_500ps.xvg") as f:
    lines = f.readlines()
data_lines = [l for l in lines if not l.startswith(("@", "#"))]

times_500 = []
rmsds_500 = []
for l in data_lines:
    parts = l.split()
    times_500.append(float(parts[0]))
    rmsds_500.append(float(parts[1]))

for i in range(0, len(times_500), 20):
    print(f"  t={times_500[i]:.1f} ps: RMSD={rmsds_500[i]:.4f} nm")
print(f"  t={times_500[-1]:.1f} ps: RMSD={rmsds_500[-1]:.4f} nm")

n = len(rmsds_500)
half1 = rmsds_500[:n//2]
half2 = rmsds_500[n//2:]

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, subprocess, shutil

os.environ["PATH"] = "/usr/local/miniconda/bin:" + os.environ["PATH"]

if not os.path.exists("/usr/local/miniconda/bin/conda"):
    if os.path.exists("/usr/local/miniconda"):
        shutil.rmtree("/usr/local/miniconda")
    !wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /content/miniconda.sh
    !bash /content/miniconda.sh -b -p /usr/local/miniconda

gmx_bin = "/usr/local/miniconda/envs/gromacs/bin.AVX2_256/gmx"
if not os.path.exists(gmx_bin):
    !conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
    !conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
    !conda config --add channels defaults
    !conda config --add channels bioconda
    !conda config --add channels conda-forge
    !conda create -n gromacs -c bioconda -c conda-forge gromacs -y

dst_ff = "/usr/local/miniconda/envs/gromacs/share/gromacs/top/amber_na.ff"
if not os.path.exists(dst_ff):
    if os.path.exists("/content/srnas_ff"):
        shutil.rmtree("/content/srnas_ff")
    subprocess.run(["git", "clone", "https://github.com/srnas/ff.git", "/content/srnas_ff"], check=True)
    shutil.copytree("/content/srnas_ff/amber_na.ff", dst_ff)

test_dir = "/content/drive/MyDrive/NEAT1_project/MD_simulation/test_nomsa"
os.chdir(test_dir)

In [ ]:
result_hbond_500 = subprocess.run(
    [gmx_bin, "hbond", "-s", "production_ext.tpr", "-f", "production_ext_500ps_whole.xtc",
     "-n", "index_regions.ndx", "-num", "hbond_count_500ps.xvg"],
    input="11\n12\n",
    capture_output=True, text=True
)

print("STDOUT:")
print(result_hbond_500.stdout[-2000:])
print("\nReturn code:", result_hbond_500.returncode)

In [ ]:
with open("hbond_count_500ps.xvg") as f:
    lines = f.readlines()

data_lines = [l for l in lines if not l.startswith(("@", "#"))]

times_hb500 = []
hbond_counts500 = []
for l in data_lines:
    parts = l.split()
    times_hb500.append(float(parts[0]))
    hbond_counts500.append(int(float(parts[1])))

for i in range(0, len(times_hb500), 25):
    print(f"  t={times_hb500[i]:.1f} ps: hbonds={hbond_counts500[i]}")
print(f"  t={times_hb500[-1]:.1f} ps: hbonds={hbond_counts500[-1]}")

n = len(hbond_counts500)
half1 = hbond_counts500[:n//2]
half2 = hbond_counts500[n//2:]